# E2 — Jalur B: k = 10 arsitektur (uji Spearman rho)

**Satu model per sesi.** Ganti `BASELINE_NAMES` di sel konfigurasi:
`["unetpp"]` -> `["resunet"]` -> `["transunet"]`.

Protokol/fold/seed **tidak boleh diubah** — itu yang membuat peringkatnya sah.

Setelah semua selesai, jalankan `R2_baseline_ranking.py` secara lokal.
**Bersiaplah untuk kemungkinan rho ternyata TINGGI** — artinya tesis antar-metode
gugur dan judul harus di-reframe. Itu justru alasan eksperimen ini dijalankan.


# BASELINE COMPARISON — BC-VMamba (+L_cimt) vs pustaka pembanding · pipeline E0

**Versi READY (Experiment baru 2).** Data-loading di-swap ke pipeline E0: GT A1 dirasterisasi, split patient-disjoint, metadata center/SNR/morf. Menyimpan prediksi per-citra (`{val,test}_fold{f}_per_image.csv`) per model → dipakai E8/E9 & uji statistik vs **BC-VMamba (+L_cimt)** (model usulan).

Butuh dataset E0 (`master_index.csv`, `splits_5fold.json`) ter-attach (dicari otomatis via rglob).
Roster di cell konfigurasi: ubah `BASELINE_NAMES` untuk menjalankan model lain di sesi berbeda.

---

## Tujuan
Perbandingan terkontrol (apple-to-apple): tiap baseline dilatih dengan **protokol identik** BC-VMamba (data E0, split 5-fold, augmentasi, loss `SimpleLoss = 0.5·Dice + 0.5·CE`, AdamW) agar selisih murni dari arsitektur. Model usulan (**BC-VMamba +L_cimt**) dilatih terpisah di notebook E1c.

## Rencana Eksekusi (Kaggle — 1 model per sesi)
Kaggle reset 30 jam/minggu; tiap baseline 5-fold ≈ 5–6 jam → jalankan **satu model per Save Version** (ubah `BASELINE_NAMES` jadi list satu-elemen).

| Sesi | `BASELINE_NAMES` | Paradigma | Est. |
|---|---|---|---|
| 1 | `["umamba"]` | Mamba (rival langsung) | ~6 jam |
| 2 | `["swin_unet"]` | Transformer | ~6 jam |
| 3 | `["attention_unet"]` | CNN + attention | ~5 jam |

Model tambahan tersedia (opsional): `unet, unetpp, resunet, transunet, unext, segformer, vm_unet_baseline`.

## Protokol Training (identik BC-VMamba — demi fairness)
- Dataset: CUBS combined (2274 sampel CV), 5-fold patient-disjoint (E0), seed=42
- Preprocessing: CLAHE + ROI crop, resize 256×256, augmentasi sama
- Loss: `SimpleLoss = 0.5·DiceLoss + 0.5·CrossEntropy(weight=[1,3])` (base — **tanpa** L_cimt; L_cimt eksklusif model usulan)
- Optimizer: AdamW, lr=1e-4, weight_decay=1e-4 · Scheduler: LinearLR(warmup 5ep) + CosineAnnealingLR
- max 100 epoch, batch_size=8, early_stop patience=15

**Referensi model usulan (E1c):** BC-VMamba +L_cimt — Dice 0.8491, IoU 0.7385, CIMT MAE 0.1199 mm, 9.63M params.


In [ ]:
import subprocess, sys, importlib

def run(cmd, show_output=False):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if show_output:
        print(result.stdout[-800:] if result.stdout else "")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
    return result.returncode

def pkg_ok(name):
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False

print("=" * 55)
print("STEP 0: Checking environment...")
run("nvidia-smi | head -10", show_output=True)
run("pip list 2>/dev/null | grep -i torch | head -5", show_output=True)

print("=" * 55)
print("STEP 1: PyTorch cu126...")
if not pkg_ok("torch"):
    print("  Installing (~5 menit)...")
    rc = run("pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print(f"  PyTorch cu126: {'OK' if rc == 0 else 'FAILED'}")
else:
    import torch
    print(f"  Already installed: {torch.__version__} -- SKIP")

print("=" * 55)
print("STEP 2: causal_conv1d...")
if not pkg_ok("causal_conv1d"):
    rc = run("pip install -q causal-conv1d>=1.4.0 --no-build-isolation")
    if rc != 0:
        rc = run("pip install -q causal-conv1d --no-build-isolation")
    print(f"  causal_conv1d: {'OK' if rc == 0 else 'FAILED'}")
else:
    print("  Already installed -- SKIP")

print("=" * 55)
print("STEP 3: mamba_ssm...")
if not pkg_ok("mamba_ssm"):
    rc = run("pip install -q mamba-ssm --no-build-isolation")
    if rc != 0:
        rc = run("pip install -q mamba-ssm --no-deps --no-build-isolation")
    print(f"  mamba_ssm: {'OK' if rc == 0 else 'FAILED'}")
else:
    print("  Already installed -- SKIP")

print("=" * 55)
print("STEP 4: Optional (thop, timm)...")
if not pkg_ok("thop"):
    run("pip install -q thop timm")
    print("  thop + timm: installed")
else:
    print("  thop + timm already installed -- SKIP")

print("=" * 55)
print("STEP 5: Verifying mamba_ssm import...")
v = subprocess.run([sys.executable, "-c", "from mamba_ssm import Mamba; print('mamba_ssm OK')"],
                   capture_output=True, text=True)
if v.returncode == 0:
    print(" ", v.stdout.strip())
    print("\n[OK] Semua dependency siap. Lanjutkan ke cell berikutnya.")
else:
    print("  [FAILED]", v.stderr[-300:])
    print("\n[!] Coba Kernel > Restart lalu run cell ini lagi.")


In [ ]:
import os
os.makedirs("models", exist_ok=True)

with open("models/__init__.py", "w", encoding="utf-8") as _f:
    _f.write('from .unet            import UNet\nfrom .unetpp          import UNetPP\nfrom .resunet         import ResUNet\nfrom .attention_unet  import AttentionUNet\nfrom .transunet       import TransUNet\nfrom .unext           import UNeXt\n# VMUNetBaseline imported lazily in get_model() — avoids requiring mamba_ssm at import time\n\n_DEFAULTS = {\n    "unet":             dict(base=64),\n    "unetpp":           dict(base=64),\n    "resunet":          dict(base=64),\n    "attention_unet":   dict(base_c=64),\n    "transunet":        dict(embed_dim=256, depth=4, heads=4, mlp_ratio=4),\n    "unext":            dict(base=64, expansion=4),\n    "vm_unet_baseline": dict(d_state=64),\n}\n\nAVAILABLE = list(_DEFAULTS.keys())\n\ndef get_model(name: str, num_classes: int = 2, **kwargs):\n    if name not in _DEFAULTS:\n        raise ValueError(f"Unknown model \'{name}\'. Available: {AVAILABLE}")\n    params = {**_DEFAULTS[name], **kwargs}\n\n    if name == "vm_unet_baseline":\n        from .vm_unet_baseline import VMUNetBaseline\n        return VMUNetBaseline(num_classes=num_classes, **params)\n\n    registry = {\n        "unet":           lambda: UNet(num_classes=num_classes, **params),\n        "unetpp":         lambda: UNetPP(num_classes=num_classes, **params),\n        "resunet":        lambda: ResUNet(num_classes=num_classes, **params),\n        "attention_unet": lambda: AttentionUNet(num_classes=num_classes, **params),\n        "transunet":      lambda: TransUNet(num_classes=num_classes, **params),\n        "unext":          lambda: UNeXt(num_classes=num_classes, **params),\n    }\n    return registry[name]()\n')

with open("models/unet.py", "w", encoding="utf-8") as _f:
    _f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.conv(x)\n\nclass UpBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)\n        self.conv = ConvBlock(in_ch, out_ch)\n\n    def forward(self, x1, x2):\n        x1 = self.up(x1)\n        diffY = x2.size()[2] - x1.size()[2]\n        diffX = x2.size()[3] - x1.size()[3]\n        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,\n                        diffY // 2, diffY - diffY // 2])\n        return self.conv(torch.cat([x2, x1], dim=1))\n\nclass UNet(nn.Module):\n    def __init__(self, in_ch=1, num_classes=2, base=64):\n        super().__init__()\n        self.inc   = ConvBlock(in_ch, base)\n        self.down1 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base, base*2))\n        self.down2 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base*2, base*4))\n        self.down3 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base*4, base*8))\n        self.down4 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base*8, base*16))\n        self.up1   = UpBlock(base*16, base*8)\n        self.up2   = UpBlock(base*8, base*4)\n        self.up3   = UpBlock(base*4, base*2)\n        self.up4   = UpBlock(base*2, base)\n        self.outc  = nn.Conv2d(base, num_classes, kernel_size=1)\n\n    def forward(self, x):\n        x1 = self.inc(x)\n        x2 = self.down1(x1)\n        x3 = self.down2(x2)\n        x4 = self.down3(x3)\n        x5 = self.down4(x4)\n        x  = self.up1(x5, x4)\n        x  = self.up2(x, x3)\n        x  = self.up3(x, x2)\n        x  = self.up4(x, x1)\n        return self.outc(x)\n')

with open("models/unetpp.py", "w", encoding="utf-8") as _f:
    _f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\nclass UNetPP(nn.Module):\n    def __init__(self, in_ch=1, num_classes=2, base=64):\n        super().__init__()\n        nb = base\n\n        self.conv00 = ConvBlock(in_ch, nb)\n        self.conv10 = ConvBlock(nb, nb * 2)\n        self.conv20 = ConvBlock(nb * 2, nb * 4)\n        self.conv30 = ConvBlock(nb * 4, nb * 8)\n        self.conv40 = ConvBlock(nb * 8, nb * 16)\n\n        self.up01 = ConvBlock(nb + nb * 2, nb)\n        self.up11 = ConvBlock(nb * 2 + nb * 4, nb * 2)\n        self.up21 = ConvBlock(nb * 4 + nb * 8, nb * 4)\n        self.up31 = ConvBlock(nb * 8 + nb * 16, nb * 8)\n\n        self.up02 = ConvBlock(nb * (1 + 1 + 2), nb)\n        self.up12 = ConvBlock(nb * (2 + 2 + 4), nb * 2)\n        self.up22 = ConvBlock(nb * (4 + 4 + 8), nb * 4)\n\n        self.up03 = ConvBlock(nb * (1 + 1 + 1 + 2), nb)\n        self.up13 = ConvBlock(nb * (2 + 2 + 2 + 4), nb * 2)\n\n        self.up04 = ConvBlock(nb * (1 + 1 + 1 + 1 + 2), nb)\n\n        self.final = nn.Conv2d(nb, num_classes, 1)\n\n    def upsample(self, x, target):\n        return F.interpolate(x, size=target.shape[2:], mode="bilinear", align_corners=True)\n\n    def forward(self, x):\n        x00 = self.conv00(x)\n        x10 = self.conv10(F.max_pool2d(x00, 2))\n        x20 = self.conv20(F.max_pool2d(x10, 2))\n        x30 = self.conv30(F.max_pool2d(x20, 2))\n        x40 = self.conv40(F.max_pool2d(x30, 2))\n\n        x01 = self.up01(torch.cat([x00, self.upsample(x10, x00)], 1))\n        x11 = self.up11(torch.cat([x10, self.upsample(x20, x10)], 1))\n        x21 = self.up21(torch.cat([x20, self.upsample(x30, x20)], 1))\n        x31 = self.up31(torch.cat([x30, self.upsample(x40, x30)], 1))\n\n        x02 = self.up02(torch.cat([x00, x01, self.upsample(x11, x00)], 1))\n        x12 = self.up12(torch.cat([x10, x11, self.upsample(x21, x10)], 1))\n        x22 = self.up22(torch.cat([x20, x21, self.upsample(x31, x20)], 1))\n\n        x03 = self.up03(torch.cat([x00, x01, x02, self.upsample(x12, x00)], 1))\n        x13 = self.up13(torch.cat([x10, x11, x12, self.upsample(x22, x10)], 1))\n\n        x04 = self.up04(torch.cat([x00, x01, x02, x03, self.upsample(x13, x00)], 1))\n\n        return self.final(x04)\n')

with open("models/resunet.py", "w", encoding="utf-8") as _f:
    _f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ResidualBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, padding=1)\n        self.bn1      = nn.BatchNorm2d(out_ch)\n        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)\n        self.bn2      = nn.BatchNorm2d(out_ch)\n        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()\n\n    def forward(self, x):\n        r = self.shortcut(x)\n        x = F.relu(self.bn1(self.conv1(x)))\n        x = self.bn2(self.conv2(x))\n        return F.relu(x + r)\n\nclass Up(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)\n        self.conv = ResidualBlock(in_ch, out_ch)\n\n    def forward(self, x1, x2):\n        x1 = self.up(x1)\n        diffY = x2.size()[2] - x1.size()[2]\n        diffX = x2.size()[3] - x1.size()[3]\n        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,\n                        diffY // 2, diffY - diffY // 2])\n        return self.conv(torch.cat([x2, x1], dim=1))\n\nclass ResUNet(nn.Module):\n    def __init__(self, in_ch=1, num_classes=2, base=64):\n        super().__init__()\n        self.inc   = ResidualBlock(in_ch, base)\n        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base, base*2))\n        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base*2, base*4))\n        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base*4, base*8))\n        self.down4 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base*8, base*16))\n        self.up1   = Up(base*16, base*8)\n        self.up2   = Up(base*8, base*4)\n        self.up3   = Up(base*4, base*2)\n        self.up4   = Up(base*2, base)\n        self.outc  = nn.Conv2d(base, num_classes, 1)\n\n    def forward(self, x):\n        x1 = self.inc(x)\n        x2 = self.down1(x1)\n        x3 = self.down2(x2)\n        x4 = self.down3(x3)\n        x5 = self.down4(x4)\n        x  = self.up1(x5, x4)\n        x  = self.up2(x, x3)\n        x  = self.up3(x, x2)\n        x  = self.up4(x, x1)\n        return self.outc(x)\n')

with open("models/attention_unet.py", "w", encoding="utf-8") as _f:
    _f.write("import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, padding=1),\n            nn.BatchNorm2d(out_ch),\n            nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.conv(x)\n\nclass UpBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)\n        self.conv = ConvBlock(in_ch, out_ch)\n\n    def forward(self, x1, x2):\n        x1 = self.up(x1)\n        diffY = x2.size()[2] - x1.size()[2]\n        diffX = x2.size()[3] - x1.size()[3]\n        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,\n                        diffY // 2, diffY - diffY // 2])\n        return self.conv(torch.cat([x2, x1], dim=1))\n\nclass AttentionBlock(nn.Module):\n    def __init__(self, F_g, F_l, F_int):\n        super().__init__()\n        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))\n        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))\n        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, g, x):\n        if g.shape[2:] != x.shape[2:]:\n            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=True)\n        psi = self.relu(self.W_g(g) + self.W_x(x))\n        return x * self.psi(psi)\n\nclass AttentionUNet(nn.Module):\n    def __init__(self, in_ch=1, num_classes=2, base_c=64):\n        super().__init__()\n        self.inc   = ConvBlock(in_ch, base_c)\n        self.down1 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base_c, base_c * 2))\n        self.down2 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base_c * 2, base_c * 4))\n        self.down3 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base_c * 4, base_c * 8))\n        self.down4 = nn.Sequential(nn.MaxPool2d(2), ConvBlock(base_c * 8, base_c * 16))\n\n        self.att1  = AttentionBlock(F_g=base_c * 16, F_l=base_c * 8,  F_int=base_c * 4)\n        self.att2  = AttentionBlock(F_g=base_c * 8,  F_l=base_c * 4,  F_int=base_c * 2)\n        self.att3  = AttentionBlock(F_g=base_c * 4,  F_l=base_c * 2,  F_int=base_c)\n        self.att4  = AttentionBlock(F_g=base_c * 2,  F_l=base_c,      F_int=base_c // 2)\n\n        self.up1   = UpBlock(base_c * 16, base_c * 8)\n        self.up2   = UpBlock(base_c * 8,  base_c * 4)\n        self.up3   = UpBlock(base_c * 4,  base_c * 2)\n        self.up4   = UpBlock(base_c * 2,  base_c)\n\n        self.outc  = nn.Conv2d(base_c, num_classes, 1)\n\n    def forward(self, x):\n        x1 = self.inc(x)\n        x2 = self.down1(x1)\n        x3 = self.down2(x2)\n        x4 = self.down3(x3)\n        x5 = self.down4(x4)\n\n        x4 = self.att1(x5, x4)\n        x3 = self.att2(x4, x3)\n        x2 = self.att3(x3, x2)\n        x1 = self.att4(x2, x1)\n\n        x  = self.up1(x5, x4)\n        x  = self.up2(x, x3)\n        x  = self.up3(x, x2)\n        x  = self.up4(x, x1)\n        return self.outc(x)\n")

with open("models/transunet.py", "w", encoding="utf-8") as _f:
    _f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ConvStem(nn.Module):\n    def __init__(self, in_ch=1, embed_dim=256):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_ch, 64,            7, stride=2, padding=3),\n            nn.BatchNorm2d(64),   nn.ReLU(inplace=True),\n            nn.Conv2d(64,  128,             3, stride=2, padding=1),\n            nn.BatchNorm2d(128),  nn.ReLU(inplace=True),\n            nn.Conv2d(128, embed_dim // 2,  3, stride=2, padding=1),\n            nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),\n            nn.Conv2d(embed_dim // 2, embed_dim, 3, stride=2, padding=1),\n        )\n\n    def forward(self, x):\n        return self.conv(x)   # [B, embed_dim, H/16, W/16]\n\nclass ViTBlock(nn.Module):\n    def __init__(self, dim, heads=4, mlp_ratio=4, dropout=0.1):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.attn  = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)\n        self.norm2 = nn.LayerNorm(dim)\n        self.mlp   = nn.Sequential(\n            nn.Linear(dim, dim * mlp_ratio), nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(dim * mlp_ratio, dim), nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]\n        return x + self.mlp(self.norm2(x))\n\nclass DecoderBlock(nn.Module):\n    def __init__(self, in_ch, out_ch):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),\n            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\nclass TransUNet(nn.Module):\n    \"\"\"Scratch implementation: ConvStem + ViT, NO ImageNet pretrained weights.\n    Cite as "TransUNet-scratch" in paper to distinguish from official pretrained variant.\n    \"\"\"\n    def __init__(self, in_ch=1, num_classes=2, embed_dim=256, depth=4, heads=4, mlp_ratio=4):\n        super().__init__()\n        self.encoder   = ConvStem(in_ch, embed_dim)\n        num_patches    = 16 * 16\n        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))\n        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))\n        self.pos_drop  = nn.Dropout(0.1)\n        self.blocks    = nn.ModuleList([ViTBlock(embed_dim, heads=heads, mlp_ratio=mlp_ratio)\n                                        for _ in range(depth)])\n        self.norm      = nn.LayerNorm(embed_dim)\n\n        self.up1  = nn.ConvTranspose2d(embed_dim,          embed_dim // 2,  2, 2)\n        self.dec1 = DecoderBlock(embed_dim // 2,  embed_dim // 4)\n        self.up2  = nn.ConvTranspose2d(embed_dim // 4,     embed_dim // 8,  2, 2)\n        self.dec2 = DecoderBlock(embed_dim // 8,  embed_dim // 8)\n        self.up3  = nn.ConvTranspose2d(embed_dim // 8,     embed_dim // 16, 2, 2)\n        self.dec3 = DecoderBlock(embed_dim // 16, embed_dim // 16)\n        self.up4  = nn.ConvTranspose2d(embed_dim // 16,    embed_dim // 32, 2, 2)\n        self.dec4 = DecoderBlock(embed_dim // 32, embed_dim // 32)\n        self.outc = nn.Conv2d(embed_dim // 32, num_classes, 1)\n\n        nn.init.trunc_normal_(self.pos_embed, std=0.02)\n        nn.init.trunc_normal_(self.cls_token, std=0.02)\n\n    def forward(self, x):\n        x = self.encoder(x)        # [B, C, 16, 16]\n        B, C, H, W = x.shape\n        x = x.flatten(2).transpose(1, 2)\n\n        cls = self.cls_token.expand(B, -1, -1)\n        x   = torch.cat((cls, x), 1)\n        x   = self.pos_drop(x + self.pos_embed)\n\n        for blk in self.blocks:\n            x = blk(x)\n        x = self.norm(x)\n\n        x = x[:, 1:, :].transpose(1, 2).reshape(B, C, H, W)\n        x = self.dec1(self.up1(x))\n        x = self.dec2(self.up2(x))\n        x = self.dec3(self.up3(x))\n        x = self.dec4(self.up4(x))\n        return self.outc(x)\n')

with open("models/unext.py", "w", encoding="utf-8") as _f:
    _f.write('import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass ConvFFN(nn.Module):\n    def __init__(self, dim, expansion=4):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(dim, dim * expansion, 1),\n            nn.GELU(),\n            nn.Conv2d(dim * expansion, dim, 1),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\nclass UNeXt(nn.Module):\n    def __init__(self, in_ch=1, num_classes=2, base=64, expansion=4):\n        super().__init__()\n        self.enc1  = nn.Sequential(nn.Conv2d(in_ch,  base,   3, padding=1), nn.GELU())\n        self.enc2  = nn.Sequential(nn.Conv2d(base,   base*2, 3, stride=2, padding=1), nn.GELU())\n        self.enc3  = nn.Sequential(nn.Conv2d(base*2, base*4, 3, stride=2, padding=1), nn.GELU())\n        self.block = ConvFFN(base*4, expansion)\n        self.up1   = nn.ConvTranspose2d(base*4, base*2, 2, 2)\n        self.dec1  = nn.Sequential(nn.Conv2d(base*4, base*2, 3, padding=1), nn.GELU())\n        self.up2   = nn.ConvTranspose2d(base*2, base,   2, 2)\n        self.dec2  = nn.Sequential(nn.Conv2d(base*2, base,   3, padding=1), nn.GELU())\n        self.outc  = nn.Conv2d(base, num_classes, 1)\n\n    def forward(self, x):\n        x1 = self.enc1(x)\n        x2 = self.enc2(x1)\n        x3 = self.enc3(x2)\n        x4 = self.block(x3)\n        x  = self.dec1(torch.cat([self.up1(x4), x2], dim=1))\n        x  = self.dec2(torch.cat([self.up2(x),  x1], dim=1))\n        return self.outc(x)\n')

with open("models/vm_unet_baseline.py", "w", encoding="utf-8") as _f:
    _f.write('"""\nVM-UNet Baseline (baseline pembanding)\nPure Mamba SSM encoder-decoder — NO bridge, attention gates, ASPP, boundary/CIMT head, deep supervision.\nUses ResConvBlock at L1-L2/L7-L8 and PVMLayer at L3-L6.\nDepends on mamba_ssm being installed.\n"""\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ntry:\n    from mamba_ssm import Mamba\nexcept ImportError:\n    raise ImportError("mamba_ssm not installed. Run: pip install mamba-ssm causal-conv1d")\n\n\nclass ResConvBlock(nn.Module):\n    def __init__(self, in_ch: int, out_ch: int):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)\n        self.bn1   = nn.BatchNorm2d(out_ch)\n        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)\n        self.bn2   = nn.BatchNorm2d(out_ch)\n        self.skip  = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()\n        self.act   = nn.GELU()\n\n    def forward(self, x):\n        r = self.skip(x)\n        x = self.act(self.bn1(self.conv1(x)))\n        x = self.bn2(self.conv2(x))\n        return self.act(x + r)\n\n\nclass PVMLayer(nn.Module):\n    """Bidirectional 4-way Mamba scan (same implementation as main model)."""\n    def __init__(self, input_dim: int, output_dim: int,\n                 d_state: int = 64, d_conv: int = 4, expand: int = 2):\n        super().__init__()\n        if input_dim % 4 != 0:\n            raise ValueError(f"input_dim must be divisible by 4, got {input_dim}")\n        self.input_dim  = input_dim\n        self.output_dim = output_dim\n        self.norm       = nn.LayerNorm(input_dim)\n        self.mamba      = Mamba(d_model=input_dim // 4, d_state=d_state,\n                                d_conv=d_conv, expand=expand)\n        self.proj       = nn.Linear(input_dim, output_dim)\n        self.skip_scale = nn.Parameter(torch.ones(1))\n\n    def forward(self, x):\n        if x.dtype == torch.float16:\n            x = x.float()\n        B, C     = x.shape[:2]\n        img_dims = x.shape[2:]\n        n_toks   = 1\n        for d in img_dims:\n            n_toks *= d\n        x_flat = x.reshape(B, C, n_toks).transpose(-1, -2)\n        x_norm = self.norm(x_flat)\n        x1, x2, x3, x4 = torch.chunk(x_norm, 4, dim=2)\n        x_mamba = torch.cat([\n            self.mamba(x1) + self.skip_scale * x1,\n            self.mamba(x2) + self.skip_scale * x2,\n            self.mamba(x3.flip(-1)).flip(-1) + self.skip_scale * x3,\n            self.mamba(x4.flip(-1)).flip(-1) + self.skip_scale * x4,\n        ], dim=2)\n        x_mamba = self.norm(x_mamba)\n        x_mamba = self.proj(x_mamba)\n        return x_mamba.transpose(-1, -2).reshape(B, self.output_dim, *img_dims)\n\n\nclass VMUNetBaseline(nn.Module):\n    """\n    Pure VM-UNet baseline untuk perbandingan adil dengan BC-VMamba.\n    Architecture:\n      Encoder : ResConvBlock L1(1→32), ResConvBlock L2(32→64)\n              + PVMLayer L3(64→128), PVMLayer L4(128→256)\n      Bottleneck: PVMLayer (256→512)\n      Decoder : PVMLayer L3(512+256→256), PVMLayer L2(256+128→128)\n              + ResConvBlock L1(128+64→64), ResConvBlock L0(64+32→32)\n      Head    : Conv2d(32, num_classes, 1)\n    No bridge attention, no attention gates, no ASPP, no auxiliary heads.\n    """\n    def __init__(self, num_classes: int = 2, d_state: int = 64):\n        super().__init__()\n        # Encoder\n        self.enc1  = ResConvBlock(1, 32)\n        self.enc2  = ResConvBlock(32, 64)\n        self.enc3  = PVMLayer(64, 128, d_state=d_state)\n        self.enc4  = PVMLayer(128, 256, d_state=d_state)\n        self.pool  = nn.MaxPool2d(2)\n\n        # Bottleneck\n        self.bottleneck = PVMLayer(256, 512, d_state=d_state)\n\n        # Decoder — channel = skip + upsampled\n        self.up4    = nn.ConvTranspose2d(512, 256, 2, 2)\n        self.dec4   = PVMLayer(512, 256, d_state=d_state)   # 256+256 in\n\n        self.up3    = nn.ConvTranspose2d(256, 128, 2, 2)\n        self.dec3   = PVMLayer(256, 128, d_state=d_state)   # 128+128 in\n\n        self.up2    = nn.ConvTranspose2d(128, 64, 2, 2)\n        self.dec2   = ResConvBlock(128, 64)                  # 64+64 in\n\n        self.up1    = nn.ConvTranspose2d(64, 32, 2, 2)\n        self.dec1   = ResConvBlock(64, 32)                   # 32+32 in\n\n        self.head   = nn.Conv2d(32, num_classes, 1)\n\n        print(f"  [VMUNetBaseline] d_state={d_state}, num_classes={num_classes}")\n\n    def forward(self, x):\n        # Encoder\n        e1 = self.enc1(x)                  # (B, 32, 256, 256)\n        e2 = self.enc2(self.pool(e1))      # (B, 64, 128, 128)\n        e3 = self.enc3(self.pool(e2))      # (B, 128, 64, 64)\n        e4 = self.enc4(self.pool(e3))      # (B, 256, 32, 32)\n\n        # Bottleneck\n        b  = self.bottleneck(self.pool(e4))  # (B, 512, 16, 16)\n\n        # Decoder\n        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))  # (B, 256, 32, 32)\n        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))  # (B, 128, 64, 64)\n        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # (B, 64, 128, 128)\n        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # (B, 32, 256, 256)\n\n        return self.head(d1)                                   # (B, num_classes, 256, 256)\n')

print("models/ written to disk — ready to import")


In [ ]:
# ============================================================
# MODELS EXP30 (SegFormer, Swin-UNet, U-Mamba) -- TAMBAHAN
# Menulis 3 file model baru + overwrite models/__init__.py
# dengan get_model() yang sudah include ketiga model ini.
# ============================================================
import base64, os

# --- models/__init__.py (updated) ---
_init_src = 'from .unet            import UNet\nfrom .unetpp          import UNetPP\nfrom .resunet         import ResUNet\nfrom .attention_unet  import AttentionUNet\nfrom .transunet       import TransUNet\nfrom .unext           import UNeXt\nfrom .segformer       import SegFormer\nfrom .swin_unet       import SwinUNet\n# VMUNetBaseline and UMamba imported lazily (require mamba_ssm)\n\n_DEFAULTS = {\n    "unet":             dict(base=64),\n    "unetpp":           dict(base=64),\n    "resunet":          dict(base=64),\n    "attention_unet":   dict(base_c=64),\n    "transunet":        dict(embed_dim=256, depth=4, heads=4, mlp_ratio=4),\n    "unext":            dict(base=64, expansion=4),\n    "vm_unet_baseline": dict(d_state=64),\n    "segformer":        dict(),\n    "swin_unet":        dict(),\n    "umamba":           dict(d_state=16),\n}\n\nAVAILABLE = list(_DEFAULTS.keys())\n\ndef get_model(name: str, num_classes: int = 2, **kwargs):\n    if name not in _DEFAULTS:\n        raise ValueError(f"Unknown model \'{name}\'. Available: {AVAILABLE}")\n    params = {**_DEFAULTS[name], **kwargs}\n\n    if name == "vm_unet_baseline":\n        from .vm_unet_baseline import VMUNetBaseline\n        return VMUNetBaseline(num_classes=num_classes, **params)\n\n    if name == "umamba":\n        from .umamba import UMamba\n        return UMamba(in_chans=1, num_classes=num_classes, **params)\n\n    registry = {\n        "unet":           lambda: UNet(num_classes=num_classes, **params),\n        "unetpp":         lambda: UNetPP(num_classes=num_classes, **params),\n        "resunet":        lambda: ResUNet(num_classes=num_classes, **params),\n        "attention_unet": lambda: AttentionUNet(num_classes=num_classes, **params),\n        "transunet":      lambda: TransUNet(num_classes=num_classes, **params),\n        "unext":          lambda: UNeXt(num_classes=num_classes, **params),\n        "segformer":      lambda: SegFormer(in_chans=1, num_classes=num_classes),\n        "swin_unet":      lambda: SwinUNet(img_size=256, patch_size=4, in_chans=1, num_classes=num_classes),\n    }\n    return registry[name]()\n'
with open("models/__init__.py", "w", encoding="utf-8") as _f:
    _f.write(_init_src)

# --- SegFormer-B1 ---
_seg_b64 = b"IiIiClNlZ0Zvcm1lci1CMSBmb3IgMkQgbWVkaWNhbCBpbWFnZSBzZWdtZW50YXRpb24uCklucHV0OiAgKEIsIDEsIEgsIFcpICBzaW5nbGUtY2hhbm5lbCBncmF5c2NhbGUsIEg9Vz0yNTYKT3V0cHV0OiAoQiwgMiwgSCwgVykgIGJpbmFyeSBzZWdtZW50YXRpb24gbG9naXRzICgyIGNsYXNzZXMpCgpSZWZlcmVuY2U6IFhpZSBldCBhbC4sICJTZWdGb3JtZXI6IFNpbXBsZSBhbmQgRWZmaWNpZW50IERlc2lnbiBmb3IKU2VtYW50aWMgU2VnbWVudGF0aW9uIHdpdGggVHJhbnNmb3JtZXJzIiwgTmV1cklQUyAyMDIxLgoKU2VsZi1jb250YWluZWQg4oCUIG9ubHkgdXNlcyB0b3JjaCwgdG9yY2gubm4sIHRvcmNoLm5uLmZ1bmN0aW9uYWwsIG1hdGguCkV4cGVjdGVkIHBhcmFtZXRlciBjb3VudDogfjE0IE0KIiIiCgppbXBvcnQgbWF0aAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEhlbHBlcnMKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBfdG9fMnR1cGxlKHgpOgogICAgaWYgaXNpbnN0YW5jZSh4LCAobGlzdCwgdHVwbGUpKToKICAgICAgICByZXR1cm4geAogICAgcmV0dXJuICh4LCB4KQoKCmNsYXNzIERXQ29udihubi5Nb2R1bGUpOgogICAgIiIiRGVwdGgtd2lzZSAzw5czIGNvbnZvbHV0aW9uIHVzZWQgaW5zaWRlIE1peEZGTi4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0pOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZHdfY29udiA9IG5uLkNvbnYyZChkaW0sIGRpbSwgMywgMSwgMSwgZ3JvdXBzPWRpbSwgYmlhcz1UcnVlKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIEgsIFcpOgogICAgICAgIEIsIE4sIEMgPSB4LnNoYXBlCiAgICAgICAgeCA9IHgudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoQiwgQywgSCwgVykKICAgICAgICB4ID0gc2VsZi5kd19jb252KHgpCiAgICAgICAgeCA9IHguZmxhdHRlbigyKS50cmFuc3Bvc2UoMSwgMikKICAgICAgICByZXR1cm4geAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgTWl4IFRyYW5zZm9ybWVyIGJ1aWxkaW5nIGJsb2NrcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgT3ZlcmxhcFBhdGNoRW1iZWQobm4uTW9kdWxlKToKICAgICIiIgogICAgT3ZlcmxhcHBpbmcgcGF0Y2ggZW1iZWRkaW5nIHZpYSBzdHJpZGUgY29udm9sdXRpb24uCiAgICBDb252ZXJ0cyAoQiwgaW5fY2gsIEgsIFcpIOKGkiAoQiwgTiwgZW1iZWRfZGltKSB3aXRoIEhfb3V0ID0gSC9zdHJpZGUuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFucywgZW1iZWRfZGltLCBwYXRjaF9zaXplLCBzdHJpZGUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHBhdGNoX3NpemUgPSBfdG9fMnR1cGxlKHBhdGNoX3NpemUpCiAgICAgICAgc3RyaWRlICAgICA9IF90b18ydHVwbGUoc3RyaWRlKQogICAgICAgIHBhZGRpbmcgICAgPSAocGF0Y2hfc2l6ZVswXSAvLyAyLCBwYXRjaF9zaXplWzFdIC8vIDIpCiAgICAgICAgc2VsZi5wcm9qICA9IG5uLkNvbnYyZChpbl9jaGFucywgZW1iZWRfZGltLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAga2VybmVsX3NpemU9cGF0Y2hfc2l6ZSwgc3RyaWRlPXN0cmlkZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhZGRpbmc9cGFkZGluZykKICAgICAgICBzZWxmLm5vcm0gID0gbm4uTGF5ZXJOb3JtKGVtYmVkX2RpbSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4ID0gc2VsZi5wcm9qKHgpICAgICAgICAgICAgICAgICAgICAgICAgICAjIChCLCBDLCBIJywgVycpCiAgICAgICAgQiwgQywgSCwgVyA9IHguc2hhcGUKICAgICAgICB4ID0geC5mbGF0dGVuKDIpLnRyYW5zcG9zZSgxLCAyKSAgICAgICAgICAjIChCLCBIJypXJywgQykKICAgICAgICB4ID0gc2VsZi5ub3JtKHgpCiAgICAgICAgcmV0dXJuIHgsIEgsIFcKCgpjbGFzcyBFZmZpY2llbnRTZWxmQXR0ZW50aW9uKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIEVmZmljaWVudCBtdWx0aS1oZWFkIHNlbGYtYXR0ZW50aW9uIHdpdGggc3BhdGlhbCByZWR1Y3Rpb24uCiAgICBXaGVuIHNyX3JhdGlvID4gMSB0aGUga2V5L3ZhbHVlIHNlcXVlbmNlcyBhcmUgc3BhdGlhbGx5IHJlZHVjZWQgYnkgYQogICAgc3RyaWRlLXNyX3JhdGlvIGNvbnZvbHV0aW9uIGJlZm9yZSB0aGUgYXR0ZW50aW9uLgogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltLCBudW1faGVhZHMsIHNyX3JhdGlvPTEsIHFrdl9iaWFzPVRydWUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGFzc2VydCBkaW0gJSBudW1faGVhZHMgPT0gMAogICAgICAgIHNlbGYubnVtX2hlYWRzICA9IG51bV9oZWFkcwogICAgICAgIHNlbGYuaGVhZF9kaW0gICA9IGRpbSAvLyBudW1faGVhZHMKICAgICAgICBzZWxmLnNjYWxlICAgICAgPSBzZWxmLmhlYWRfZGltICoqIC0wLjUKICAgICAgICBzZWxmLnNyX3JhdGlvICAgPSBzcl9yYXRpbwoKICAgICAgICBzZWxmLnEgICA9IG5uLkxpbmVhcihkaW0sIGRpbSwgYmlhcz1xa3ZfYmlhcykKICAgICAgICBzZWxmLmt2ICA9IG5uLkxpbmVhcihkaW0sIGRpbSAqIDIsIGJpYXM9cWt2X2JpYXMpCiAgICAgICAgc2VsZi5wcm9qID0gbm4uTGluZWFyKGRpbSwgZGltKQoKICAgICAgICBpZiBzcl9yYXRpbyA+IDE6CiAgICAgICAgICAgIHNlbGYuc3IgICA9IG5uLkNvbnYyZChkaW0sIGRpbSwgc3JfcmF0aW8sIHN0cmlkZT1zcl9yYXRpbykKICAgICAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5ZXJOb3JtKGRpbSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBILCBXKToKICAgICAgICBCLCBOLCBDID0geC5zaGFwZQogICAgICAgIE5oID0gc2VsZi5udW1faGVhZHMKICAgICAgICBkICA9IHNlbGYuaGVhZF9kaW0KCiAgICAgICAgcSA9IHNlbGYucSh4KS5yZXNoYXBlKEIsIE4sIE5oLCBkKS5wZXJtdXRlKDAsIDIsIDEsIDMpICAjIChCLCBOaCwgTiwgZCkKCiAgICAgICAgaWYgc2VsZi5zcl9yYXRpbyA+IDE6CiAgICAgICAgICAgIHhfICA9IHgudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoQiwgQywgSCwgVykKICAgICAgICAgICAgeF8gID0gc2VsZi5zcih4XykuZmxhdHRlbigyKS50cmFuc3Bvc2UoMSwgMikgICAgICAgICAgIyAoQiwgTicsIEMpCiAgICAgICAgICAgIHhfICA9IHNlbGYubm9ybSh4XykKICAgICAgICBlbHNlOgogICAgICAgICAgICB4XyA9IHgKCiAgICAgICAga3YgPSBzZWxmLmt2KHhfKS5yZXNoYXBlKEIsIC0xLCAyLCBOaCwgZCkucGVybXV0ZSgyLCAwLCAzLCAxLCA0KQogICAgICAgIGssIHYgPSBrdlswXSwga3ZbMV0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGVhY2ggKEIsIE5oLCBOJywgZCkKCiAgICAgICAgYXR0biA9IChxIEAgay50cmFuc3Bvc2UoLTIsIC0xKSkgKiBzZWxmLnNjYWxlICAgICAgICAgICAgICMgKEIsIE5oLCBOLCBOJykKICAgICAgICBhdHRuID0gYXR0bi5zb2Z0bWF4KGRpbT0tMSkKCiAgICAgICAgeCA9IChhdHRuIEAgdikudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoQiwgTiwgQykKICAgICAgICB4ID0gc2VsZi5wcm9qKHgpCiAgICAgICAgcmV0dXJuIHgKCgpjbGFzcyBNaXhGRk4obm4uTW9kdWxlKToKICAgICIiIgogICAgTWl4LUZGTjogTGluZWFyIOKGkiBHRUxVIOKGkiBEV0NvbnYg4oaSIExpbmVhci4KICAgIFRoZSBkZXB0aC13aXNlIGNvbnYgbWl4ZXMgcG9zaXRpb25hbCBpbmZvcm1hdGlvbi4KICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2ZlYXR1cmVzLCBoaWRkZW5fZmVhdHVyZXMpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZmMxICAgID0gbm4uTGluZWFyKGluX2ZlYXR1cmVzLCBoaWRkZW5fZmVhdHVyZXMpCiAgICAgICAgc2VsZi5kd19jb252ID0gRFdDb252KGhpZGRlbl9mZWF0dXJlcykKICAgICAgICBzZWxmLmZjMiAgICA9IG5uLkxpbmVhcihoaWRkZW5fZmVhdHVyZXMsIGluX2ZlYXR1cmVzKQogICAgICAgIHNlbGYuYWN0ICAgID0gbm4uR0VMVSgpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCwgSCwgVyk6CiAgICAgICAgeCA9IHNlbGYuZmMxKHgpCiAgICAgICAgeCA9IHNlbGYuYWN0KHgpCiAgICAgICAgeCA9IHNlbGYuZHdfY29udih4LCBILCBXKQogICAgICAgIHggPSBzZWxmLmZjMih4KQogICAgICAgIHJldHVybiB4CgoKY2xhc3MgTWlUQmxvY2sobm4uTW9kdWxlKToKICAgICIiIgogICAgT25lIE1peCBUcmFuc2Zvcm1lciBibG9jazoKICAgICAgICB4ID0geCArIEF0dG4oTm9ybSh4KSkKICAgICAgICB4ID0geCArIEZGTihOb3JtKHgpKQogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGltLCBudW1faGVhZHMsIG1scF9yYXRpbz00LCBzcl9yYXRpbz0xKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5vcm0xID0gbm4uTGF5ZXJOb3JtKGRpbSkKICAgICAgICBzZWxmLmF0dG4gID0gRWZmaWNpZW50U2VsZkF0dGVudGlvbihkaW0sIG51bV9oZWFkcywgc3JfcmF0aW89c3JfcmF0aW8pCiAgICAgICAgc2VsZi5ub3JtMiA9IG5uLkxheWVyTm9ybShkaW0pCiAgICAgICAgc2VsZi5mZm4gICA9IE1peEZGTihkaW0sIGludChkaW0gKiBtbHBfcmF0aW8pKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIEgsIFcpOgogICAgICAgIHggPSB4ICsgc2VsZi5hdHRuKHNlbGYubm9ybTEoeCksIEgsIFcpCiAgICAgICAgeCA9IHggKyBzZWxmLmZmbihzZWxmLm5vcm0yKHgpLCBILCBXKQogICAgICAgIHJldHVybiB4CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBNaXggVHJhbnNmb3JtZXIgZW5jb2RlciAoYmFja2JvbmUpCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpjbGFzcyBNaXhUcmFuc2Zvcm1lcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICA0LXN0YWdlIE1peCBUcmFuc2Zvcm1lciBlbmNvZGVyIChTZWdGb3JtZXItQjEgY29uZmlndXJhdGlvbikuCgogICAgU3RhZ2UgZGltcyA6IFs2NCwgMTI4LCAzMjAsIDUxMl0KICAgIE51bSBoZWFkcyAgOiBbMSwgIDIsICAgNSwgICA4XQogICAgU1IgcmF0aW9zICA6IFs4LCAgNCwgICAyLCAgIDFdCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFucz0xKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBlbWJlZF9kaW1zICA9IFs2NCwgMTI4LCAzMjAsIDUxMl0KICAgICAgICBudW1faGVhZHMgICA9IFsxLCAgMiwgICA1LCAgIDhdCiAgICAgICAgc3JfcmF0aW9zICAgPSBbOCwgIDQsICAgMiwgICAxXQogICAgICAgIGRlcHRocyAgICAgID0gWzIsICAyLCAgIDIsICAgMl0KICAgICAgICBtbHBfcmF0aW9zICA9IFs0LCAgNCwgICA0LCAgIDRdCgogICAgICAgICMgUGF0Y2ggZW1iZWRkaW5ncwogICAgICAgIHNlbGYucGF0Y2hfZW1iZWQxID0gT3ZlcmxhcFBhdGNoRW1iZWQoaW5fY2hhbnMsICAgICAgICBlbWJlZF9kaW1zWzBdLCBwYXRjaF9zaXplPTcsIHN0cmlkZT00KQogICAgICAgIHNlbGYucGF0Y2hfZW1iZWQyID0gT3ZlcmxhcFBhdGNoRW1iZWQoZW1iZWRfZGltc1swXSwgICBlbWJlZF9kaW1zWzFdLCBwYXRjaF9zaXplPTMsIHN0cmlkZT0yKQogICAgICAgIHNlbGYucGF0Y2hfZW1iZWQzID0gT3ZlcmxhcFBhdGNoRW1iZWQoZW1iZWRfZGltc1sxXSwgICBlbWJlZF9kaW1zWzJdLCBwYXRjaF9zaXplPTMsIHN0cmlkZT0yKQogICAgICAgIHNlbGYucGF0Y2hfZW1iZWQ0ID0gT3ZlcmxhcFBhdGNoRW1iZWQoZW1iZWRfZGltc1syXSwgICBlbWJlZF9kaW1zWzNdLCBwYXRjaF9zaXplPTMsIHN0cmlkZT0yKQoKICAgICAgICAjIFRyYW5zZm9ybWVyIGJsb2NrcyBwZXIgc3RhZ2UKICAgICAgICBzZWxmLmJsb2NrMSA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBNaVRCbG9jayhlbWJlZF9kaW1zWzBdLCBudW1faGVhZHNbMF0sIG1scF9yYXRpb3NbMF0sIHNyX3JhdGlvc1swXSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoZGVwdGhzWzBdKQogICAgICAgIF0pCiAgICAgICAgc2VsZi5ub3JtMSA9IG5uLkxheWVyTm9ybShlbWJlZF9kaW1zWzBdKQoKICAgICAgICBzZWxmLmJsb2NrMiA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBNaVRCbG9jayhlbWJlZF9kaW1zWzFdLCBudW1faGVhZHNbMV0sIG1scF9yYXRpb3NbMV0sIHNyX3JhdGlvc1sxXSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoZGVwdGhzWzFdKQogICAgICAgIF0pCiAgICAgICAgc2VsZi5ub3JtMiA9IG5uLkxheWVyTm9ybShlbWJlZF9kaW1zWzFdKQoKICAgICAgICBzZWxmLmJsb2NrMyA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBNaVRCbG9jayhlbWJlZF9kaW1zWzJdLCBudW1faGVhZHNbMl0sIG1scF9yYXRpb3NbMl0sIHNyX3JhdGlvc1syXSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoZGVwdGhzWzJdKQogICAgICAgIF0pCiAgICAgICAgc2VsZi5ub3JtMyA9IG5uLkxheWVyTm9ybShlbWJlZF9kaW1zWzJdKQoKICAgICAgICBzZWxmLmJsb2NrNCA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBNaVRCbG9jayhlbWJlZF9kaW1zWzNdLCBudW1faGVhZHNbM10sIG1scF9yYXRpb3NbM10sIHNyX3JhdGlvc1szXSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoZGVwdGhzWzNdKQogICAgICAgIF0pCiAgICAgICAgc2VsZi5ub3JtNCA9IG5uLkxheWVyTm9ybShlbWJlZF9kaW1zWzNdKQoKICAgICAgICBzZWxmLl9pbml0X3dlaWdodHMoKQoKICAgIGRlZiBfaW5pdF93ZWlnaHRzKHNlbGYpOgogICAgICAgIGZvciBtIGluIHNlbGYubW9kdWxlcygpOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKG0sIG5uLkxpbmVhcik6CiAgICAgICAgICAgICAgICBubi5pbml0LnRydW5jX25vcm1hbF8obS53ZWlnaHQsIHN0ZD0wLjAyKQogICAgICAgICAgICAgICAgaWYgbS5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG0uYmlhcykKICAgICAgICAgICAgZWxpZiBpc2luc3RhbmNlKG0sIG5uLkxheWVyTm9ybSk6CiAgICAgICAgICAgICAgICBubi5pbml0Lm9uZXNfKG0ud2VpZ2h0KQogICAgICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obS5iaWFzKQogICAgICAgICAgICBlbGlmIGlzaW5zdGFuY2UobSwgbm4uQ29udjJkKToKICAgICAgICAgICAgICAgIG5uLmluaXQua2FpbWluZ19ub3JtYWxfKG0ud2VpZ2h0LCBtb2RlPSdmYW5fb3V0JykKICAgICAgICAgICAgICAgIGlmIG0uYmlhcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBubi5pbml0Lnplcm9zXyhtLmJpYXMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgb3V0cyA9IFtdCgogICAgICAgICMgU3RhZ2UgMQogICAgICAgIHgsIEgsIFcgPSBzZWxmLnBhdGNoX2VtYmVkMSh4KQogICAgICAgIGZvciBibGsgaW4gc2VsZi5ibG9jazE6CiAgICAgICAgICAgIHggPSBibGsoeCwgSCwgVykKICAgICAgICB4ID0gc2VsZi5ub3JtMSh4KQogICAgICAgIG91dHMuYXBwZW5kKHgucmVzaGFwZSh4LnNoYXBlWzBdLCBILCBXLCAtMSkucGVybXV0ZSgwLCAzLCAxLCAyKSkKCiAgICAgICAgIyBTdGFnZSAyCiAgICAgICAgeCA9IG91dHNbLTFdCiAgICAgICAgeCwgSCwgVyA9IHNlbGYucGF0Y2hfZW1iZWQyKHgpCiAgICAgICAgZm9yIGJsayBpbiBzZWxmLmJsb2NrMjoKICAgICAgICAgICAgeCA9IGJsayh4LCBILCBXKQogICAgICAgIHggPSBzZWxmLm5vcm0yKHgpCiAgICAgICAgb3V0cy5hcHBlbmQoeC5yZXNoYXBlKHguc2hhcGVbMF0sIEgsIFcsIC0xKS5wZXJtdXRlKDAsIDMsIDEsIDIpKQoKICAgICAgICAjIFN0YWdlIDMKICAgICAgICB4ID0gb3V0c1stMV0KICAgICAgICB4LCBILCBXID0gc2VsZi5wYXRjaF9lbWJlZDMoeCkKICAgICAgICBmb3IgYmxrIGluIHNlbGYuYmxvY2szOgogICAgICAgICAgICB4ID0gYmxrKHgsIEgsIFcpCiAgICAgICAgeCA9IHNlbGYubm9ybTMoeCkKICAgICAgICBvdXRzLmFwcGVuZCh4LnJlc2hhcGUoeC5zaGFwZVswXSwgSCwgVywgLTEpLnBlcm11dGUoMCwgMywgMSwgMikpCgogICAgICAgICMgU3RhZ2UgNAogICAgICAgIHggPSBvdXRzWy0xXQogICAgICAgIHgsIEgsIFcgPSBzZWxmLnBhdGNoX2VtYmVkNCh4KQogICAgICAgIGZvciBibGsgaW4gc2VsZi5ibG9jazQ6CiAgICAgICAgICAgIHggPSBibGsoeCwgSCwgVykKICAgICAgICB4ID0gc2VsZi5ub3JtNCh4KQogICAgICAgIG91dHMuYXBwZW5kKHgucmVzaGFwZSh4LnNoYXBlWzBdLCBILCBXLCAtMSkucGVybXV0ZSgwLCAzLCAxLCAyKSkKCiAgICAgICAgcmV0dXJuIG91dHMgICAjIGxpc3Qgb2YgNCBmZWF0dXJlIG1hcHMsIHN0cmlkZXMgNCwgOCwgMTYsIDMyCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBBbGwtTUxQIERlY29kZSBIZWFkCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpjbGFzcyBTZWdGb3JtZXJIZWFkKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIEFsbC1NTFAgZGVjb2RlciBoZWFkLgogICAgUHJvamVjdHMgZWFjaCBzdGFnZSBvdXRwdXQgdG8gZW1iZWRfZGltPTI1NiwgdXBzYW1wbGVzIGFsbCB0byAxLzQgcmVzb2x1dGlvbiwKICAgIGNvbmNhdGVuYXRlcywgZnVzZXMsIGFuZCBwcm9kdWNlcyBmaW5hbCBsb2dpdHMuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFubmVscywgZW1iZWRfZGltPTI1NiwgbnVtX2NsYXNzZXM9Mik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5saW5lYXJfbGF5ZXJzID0gbm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgIG5uLkxpbmVhcihjLCBlbWJlZF9kaW0pIGZvciBjIGluIGluX2NoYW5uZWxzCiAgICAgICAgXSkKICAgICAgICBzZWxmLmxpbmVhcl9mdXNlID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKGVtYmVkX2RpbSAqIGxlbihpbl9jaGFubmVscyksIGVtYmVkX2RpbSwgMSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGVtYmVkX2RpbSksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCiAgICAgICAgc2VsZi5saW5lYXJfcHJlZCA9IG5uLkNvbnYyZChlbWJlZF9kaW0sIG51bV9jbGFzc2VzLCAxKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGZlYXR1cmVzLCBpbWdfaCwgaW1nX3cpOgogICAgICAgIHRhcmdldF9oID0gaW1nX2ggLy8gNAogICAgICAgIHRhcmdldF93ID0gaW1nX3cgLy8gNAogICAgICAgIHByb2plY3RlZCA9IFtdCiAgICAgICAgZm9yIGksIGZlYXQgaW4gZW51bWVyYXRlKGZlYXR1cmVzKToKICAgICAgICAgICAgQiwgQywgSCwgVyA9IGZlYXQuc2hhcGUKICAgICAgICAgICAgIyBMaW5lYXIgcHJvamVjdGlvbjogcmVzaGFwZSB0byB0b2tlbnMsIHByb2plY3QsIHJlc2hhcGUgYmFjawogICAgICAgICAgICBmZWF0X2ZsYXQgPSBmZWF0LmZsYXR0ZW4oMikudHJhbnNwb3NlKDEsIDIpICAgIyAoQiwgSCpXLCBDKQogICAgICAgICAgICBmZWF0X2ZsYXQgPSBzZWxmLmxpbmVhcl9sYXllcnNbaV0oZmVhdF9mbGF0KSAgICMgKEIsIEgqVywgZW1iZWRfZGltKQogICAgICAgICAgICBmZWF0XzJkICAgPSBmZWF0X2ZsYXQudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoQiwgLTEsIEgsIFcpCiAgICAgICAgICAgIGlmIEggIT0gdGFyZ2V0X2ggb3IgVyAhPSB0YXJnZXRfdzoKICAgICAgICAgICAgICAgIGZlYXRfMmQgPSBGLmludGVycG9sYXRlKGZlYXRfMmQsIHNpemU9KHRhcmdldF9oLCB0YXJnZXRfdyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgICAgIHByb2plY3RlZC5hcHBlbmQoZmVhdF8yZCkKCiAgICAgICAgeCA9IHRvcmNoLmNhdChwcm9qZWN0ZWQsIGRpbT0xKSAgICAjIChCLCBlbWJlZF9kaW0qNCwgSC80LCBXLzQpCiAgICAgICAgeCA9IHNlbGYubGluZWFyX2Z1c2UoeCkKICAgICAgICB4ID0gc2VsZi5saW5lYXJfcHJlZCh4KQogICAgICAgIHggPSBGLmludGVycG9sYXRlKHgsIHNpemU9KGltZ19oLCBpbWdfdyksIG1vZGU9J2JpbGluZWFyJywgYWxpZ25fY29ybmVycz1GYWxzZSkKICAgICAgICByZXR1cm4geAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRnVsbCBtb2RlbAojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgU2VnRm9ybWVyKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFNlZ0Zvcm1lci1CMSBmb3IgYmluYXJ5IG1lZGljYWwgaW1hZ2Ugc2VnbWVudGF0aW9uLgoKICAgIElucHV0IDogKEIsIDEsIDI1NiwgMjU2KQogICAgT3V0cHV0OiAoQiwgMiwgMjU2LCAyNTYpCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFucz0xLCBudW1fY2xhc3Nlcz0yKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmJhY2tib25lICA9IE1peFRyYW5zZm9ybWVyKGluX2NoYW5zPWluX2NoYW5zKQogICAgICAgIGluX2NoYW5uZWxzICAgID0gWzY0LCAxMjgsIDMyMCwgNTEyXQogICAgICAgIHNlbGYuZGVjb2RlX2hlYWQgPSBTZWdGb3JtZXJIZWFkKGluX2NoYW5uZWxzLCBlbWJlZF9kaW09MjU2LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIF8sIF8sIEgsIFcgPSB4LnNoYXBlCiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmJhY2tib25lKHgpCiAgICAgICAgcmV0dXJuIHNlbGYuZGVjb2RlX2hlYWQoZmVhdHVyZXMsIEgsIFcpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBRdWljayBzYW5pdHkgY2hlY2sKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBtb2RlbCA9IFNlZ0Zvcm1lcihpbl9jaGFucz0xLCBudW1fY2xhc3Nlcz0yKQogICAgbl9wYXJhbXMgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHByaW50KGYnU2VnRm9ybWVyLUIxICBwYXJhbWV0ZXJzOiB7bl9wYXJhbXMgLyAxZTY6LjFmfSBNJykKICAgIHggPSB0b3JjaC5yYW5kbigyLCAxLCAyNTYsIDI1NikKICAgIHkgPSBtb2RlbCh4KQogICAgcHJpbnQoZidJbnB1dDoge3guc2hhcGV9ICDihpIgIE91dHB1dDoge3kuc2hhcGV9JykKICAgIGFzc2VydCB5LnNoYXBlID09ICgyLCAyLCAyNTYsIDI1NiksIGYnVW5leHBlY3RlZCBvdXRwdXQgc2hhcGU6IHt5LnNoYXBlfScKICAgIHByaW50KCdPSycpCg=="
with open("models/segformer.py", "wb") as _f:
    _f.write(base64.b64decode(_seg_b64))

# --- Swin-UNet-Tiny ---
_swi_b64 = b"..."
with open("models/swin_unet.py", "wb") as _f:
    _f.write(base64.b64decode(_swi_b64))

# --- U-Mamba (standalone 2D) ---
_uma_b64 = b"IiIiClUtTWFtYmEgKHN0YW5kYWxvbmUgMkQgYWRhcHRhdGlvbikgZm9yIG1lZGljYWwgaW1hZ2Ugc2VnbWVudGF0aW9uLgpJbnB1dDogIChCLCAxLCBILCBXKSAgc2luZ2xlLWNoYW5uZWwgZ3JheXNjYWxlLCBIPVc9MjU2Ck91dHB1dDogKEIsIDIsIEgsIFcpICBiaW5hcnkgc2VnbWVudGF0aW9uIGxvZ2l0cyAoMiBjbGFzc2VzKQoKUmVmZXJlbmNlOiBNYSBldCBhbC4sICJVLU1hbWJhOiBFbmhhbmNpbmcgTG9uZy1yYW5nZSBEZXBlbmRlbmN5IGZvcgpCaW9tZWRpY2FsIEltYWdlIFNlZ21lbnRhdGlvbiIsIGFyWGl2IDIwMjQuCgpUaGlzIGlzIGEgc3RhbmRhbG9uZSAyRCBhZGFwdGF0aW9uIHRoYXQgZG9lcyBOT1QgZGVwZW5kIG9uIG5uVU5ldC4KSXQgcmVxdWlyZXMgdGhlIG1hbWJhX3NzbSBwYWNrYWdlIChwaXAgaW5zdGFsbCBtYW1iYS1zc20pLgpBcmNoaXRlY3R1cmUgbWlycm9ycyB0aGUgZW5jb2RlcuKAk2JvdHRsZW5lY2vigJNkZWNvZGVyIHN0eWxlIG9mIFUtTWFtYmEgd2l0aApSZXNDb252QmxvY2sgKyBNYW1iYUxheWVyIHBlciBzdGFnZSBhbmQgc2tpcCBjb25uZWN0aW9ucyBpbiB0aGUgZGVjb2Rlci4KClNlbGYtY29udGFpbmVkIOKAlCBvbmx5IHVzZXMgdG9yY2gsIHRvcmNoLm5uLCB0b3JjaC5ubi5mdW5jdGlvbmFsLCBtYXRoLAphbmQgbWFtYmFfc3NtIChsYXp5LWltcG9ydGVkIHdpdGggYSBjbGVhciBlcnJvciBpZiBhYnNlbnQpLgoKUGFyYW1ldGVyIGNvdW50OiB+MTAgTSB3aXRoIHJlYWwgbWFtYmFfc3NtIChjaGFubmVscz1bMzIsNjQsMTI4LDI1Nl0pLgpUaGUgIn4yME0iIGZpZ3VyZSBpbiBzb21lIHJlZmVyZW5jZXMgYXNzdW1lcyBsYXJnZXIgY2hhbm5lbHMgKGUuZy4gWzQ4LDk2LDE5MiwzODRdKS4KVGhlIGNoYW5uZWxzIGhlcmUgbWF0Y2ggdGhlIHNwZWMgKFszMiw2NCwxMjgsMjU2XSkgYW5kIHByb2R1Y2UgYSBsaWdodGVyIGJ1dAphcmNoaXRlY3R1cmFsbHkgY29ycmVjdCBtb2RlbC4gIFRvIHNjYWxlIHVwLCBwYXNzIGVuY19jaGFubmVscz1bNDgsOTYsMTkyLDM4NF0uCiIiIgoKaW1wb3J0IG1hdGgKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBMYXp5IGltcG9ydCBvZiBtYW1iYV9zc20KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmRlZiBfZ2V0X21hbWJhKCk6CiAgICB0cnk6CiAgICAgICAgZnJvbSBtYW1iYV9zc20gaW1wb3J0IE1hbWJhCiAgICAgICAgcmV0dXJuIE1hbWJhCiAgICBleGNlcHQgSW1wb3J0RXJyb3IgYXMgZToKICAgICAgICByYWlzZSBJbXBvcnRFcnJvcigKICAgICAgICAgICAgIm1hbWJhX3NzbSBpcyByZXF1aXJlZCBmb3IgVS1NYW1iYS4gIgogICAgICAgICAgICAiSW5zdGFsbCBpdCB3aXRoOiBwaXAgaW5zdGFsbCBtYW1iYS1zc20iCiAgICAgICAgKSBmcm9tIGUKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEJ1aWxkaW5nIGJsb2NrcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKY2xhc3MgUmVzQ29udkJsb2NrKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFJlc2lkdWFsIGNvbnZvbHV0aW9uYWwgYmxvY2s6CiAgICAgIENvbnYzw5czIOKGkiBCTiDihpIgR0VMVSDihpIgQ29udjPDlzMg4oaSIEJOIOKGkiBHRUxVCiAgICAgICsgc2tpcCAoQ29udjHDlzEgKyBCTiBpZiBpbl9jaCDiiaAgb3V0X2NoKQogICAgIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2gsIG91dF9jaCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb252MSA9IG5uLkNvbnYyZChpbl9jaCwgIG91dF9jaCwgMywgcGFkZGluZz0xLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4xICAgPSBubi5CYXRjaE5vcm0yZChvdXRfY2gpCiAgICAgICAgc2VsZi5jb252MiA9IG5uLkNvbnYyZChvdXRfY2gsIG91dF9jaCwgMywgcGFkZGluZz0xLCBiaWFzPUZhbHNlKQogICAgICAgIHNlbGYuYm4yICAgPSBubi5CYXRjaE5vcm0yZChvdXRfY2gpCiAgICAgICAgc2VsZi5hY3QgICA9IG5uLkdFTFUoKQoKICAgICAgICBpZiBpbl9jaCAhPSBvdXRfY2g6CiAgICAgICAgICAgIHNlbGYuc2tpcCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBubi5Db252MmQoaW5fY2gsIG91dF9jaCwgMSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgICAgICBubi5CYXRjaE5vcm0yZChvdXRfY2gpLAogICAgICAgICAgICApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5za2lwID0gbm4uSWRlbnRpdHkoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJlc2lkdWFsID0gc2VsZi5za2lwKHgpCiAgICAgICAgeCA9IHNlbGYuYWN0KHNlbGYuYm4xKHNlbGYuY29udjEoeCkpKQogICAgICAgIHggPSBzZWxmLmFjdChzZWxmLmJuMihzZWxmLmNvbnYyKHgpKSkKICAgICAgICByZXR1cm4geCArIHJlc2lkdWFsCgoKY2xhc3MgTWFtYmFMYXllcihubi5Nb2R1bGUpOgogICAgIiIiCiAgICBTZXF1ZW5jZSBNYW1iYSBsYXllciBhcHBsaWVkIHRvIHNwYXRpYWwgZmVhdHVyZXM6CiAgICAgIDEuIEZsYXR0ZW4gKEgsIFcpIGludG8gc2VxdWVuY2UgZGltZW5zaW9uIEwgPSBIKlcKICAgICAgMi4gTGF5ZXJOb3JtIG9uIGNoYW5uZWwgZGltCiAgICAgIDMuIE1hbWJhIFNTTSAoZF9tb2RlbCA9IEMpCiAgICAgIDQuIFJlc2lkdWFsIGFkZAogICAgICA1LiBSZXNoYXBlIGJhY2sgdG8gKEIsIEMsIEgsIFcpCgogICAgTWFtYmEgcHJvY2Vzc2VzIChCLCBMLCBDKSBzZXF1ZW5jZXMuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0sIGRfc3RhdGU9MTYsIGRfY29udj00LCBleHBhbmQ9Mik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ub3JtICA9IG5uLkxheWVyTm9ybShkaW0pCiAgICAgICAgTWFtYmEgPSBfZ2V0X21hbWJhKCkKICAgICAgICBzZWxmLm1hbWJhID0gTWFtYmEoCiAgICAgICAgICAgIGRfbW9kZWw9ZGltLAogICAgICAgICAgICBkX3N0YXRlPWRfc3RhdGUsCiAgICAgICAgICAgIGRfY29udj1kX2NvbnYsCiAgICAgICAgICAgIGV4cGFuZD1leHBhbmQsCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIEIsIEMsIEgsIFcgPSB4LnNoYXBlCiAgICAgICAgIyBGbGF0dGVuIHNwYXRpYWwgdG8gc2VxdWVuY2U6IChCLCBMLCBDKQogICAgICAgIHhfc2VxID0geC5mbGF0dGVuKDIpLnRyYW5zcG9zZSgxLCAyKSAgICAgICAgICAjIChCLCBIKlcsIEMpCiAgICAgICAgIyBBcHBseSBMYXllck5vcm0gKyBNYW1iYSB3aXRoIHJlc2lkdWFsCiAgICAgICAgeF9ub3JtID0gc2VsZi5ub3JtKHhfc2VxKQogICAgICAgIHhfb3V0ICA9IHNlbGYubWFtYmEoeF9ub3JtKSArIHhfc2VxICAgICAgICAgICAgIyAoQiwgSCpXLCBDKQogICAgICAgICMgUmVzaGFwZSBiYWNrCiAgICAgICAgeF9vdXQgID0geF9vdXQudHJhbnNwb3NlKDEsIDIpLnJlc2hhcGUoQiwgQywgSCwgVykKICAgICAgICByZXR1cm4geF9vdXQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEVuY29kZXIgc3RhZ2UKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmNsYXNzIEVuY29kZXJTdGFnZShubi5Nb2R1bGUpOgogICAgIiIiT25lIGVuY29kZXIgc3RhZ2U6IFJlc0NvbnZCbG9jayArIE1hbWJhTGF5ZXIgKG5vIHBvb2xpbmcgaGVyZSkuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2gsIG91dF9jaCwgZF9zdGF0ZT0xNiwgZF9jb252PTQsIGV4cGFuZD0yKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnJlc19ibG9jayAgID0gUmVzQ29udkJsb2NrKGluX2NoLCBvdXRfY2gpCiAgICAgICAgc2VsZi5tYW1iYV9sYXllciA9IE1hbWJhTGF5ZXIob3V0X2NoLCBkX3N0YXRlPWRfc3RhdGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZF9jb252PWRfY29udiwgZXhwYW5kPWV4cGFuZCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4ID0gc2VsZi5yZXNfYmxvY2soeCkKICAgICAgICB4ID0gc2VsZi5tYW1iYV9sYXllcih4KQogICAgICAgIHJldHVybiB4CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBEZWNvZGVyIHN0YWdlCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpjbGFzcyBEZWNvZGVyU3RhZ2Uobm4uTW9kdWxlKToKICAgICIiIgogICAgT25lIGRlY29kZXIgc3RhZ2U6CiAgICAgIDEuIENvbnZUcmFuc3Bvc2UyZCB0byB1cHNhbXBsZSBhbmQgcmVkdWNlIGNoYW5uZWxzICh1cF9jaCDihpIgc2tpcF9jaCkKICAgICAgMi4gQ29uY2F0ZW5hdGUgd2l0aCBza2lwIGNvbm5lY3Rpb24KICAgICAgMy4gUmVzQ29udkJsb2NrICh1cF9jaCArIHNraXBfY2gg4oaSIG91dF9jaCkKICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHVwX2NoLCBza2lwX2NoLCBvdXRfY2gpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYudXAgICAgICAgID0gbm4uQ29udlRyYW5zcG9zZTJkKHVwX2NoLCBza2lwX2NoLCBrZXJuZWxfc2l6ZT0yLCBzdHJpZGU9MikKICAgICAgICBzZWxmLnJlc19ibG9jayA9IFJlc0NvbnZCbG9jayhza2lwX2NoICsgc2tpcF9jaCwgb3V0X2NoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIHNraXApOgogICAgICAgIHggPSBzZWxmLnVwKHgpCiAgICAgICAgIyBQYWQgaWYgc3BhdGlhbCBzaXplcyBkaWZmZXIgKHNob3VsZG4ndCBoYXBwZW4gd2l0aCAyNTYgaW5wdXQsIGJ1dCBzYWZlKQogICAgICAgIGlmIHguc2hhcGVbLTI6XSAhPSBza2lwLnNoYXBlWy0yOl06CiAgICAgICAgICAgIHggPSBGLmludGVycG9sYXRlKHgsIHNpemU9c2tpcC5zaGFwZVstMjpdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9RmFsc2UpCiAgICAgICAgeCA9IHRvcmNoLmNhdChbeCwgc2tpcF0sIGRpbT0xKQogICAgICAgIHggPSBzZWxmLnJlc19ibG9jayh4KQogICAgICAgIHJldHVybiB4CgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBGdWxsIFUtTWFtYmEgbW9kZWwKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCmNsYXNzIFVNYW1iYShubi5Nb2R1bGUpOgogICAgIiIiCiAgICBVLU1hbWJhIDJEIHN0YW5kYWxvbmUgYWRhcHRhdGlvbiBmb3IgYmluYXJ5IG1lZGljYWwgaW1hZ2Ugc2VnbWVudGF0aW9uLgoKICAgIEVuY29kZXIgY2hhbm5lbHMgOiBbMzIsIDY0LCAxMjgsIDI1Nl0KICAgIEJvdHRsZW5lY2sgICAgICAgOiA1MTIKICAgIERlY29kZXIgY2hhbm5lbHMgOiBtaXJyb3Igb2YgZW5jb2RlcgogICAgSGVhZCAgICAgICAgICAgICA6IENvbnYyZCgzMiwgbnVtX2NsYXNzZXMsIDEpCgogICAgSW5wdXQgOiAoQiwgMSwgMjU2LCAyNTYpCiAgICBPdXRwdXQ6IChCLCAyLCAyNTYsIDI1NikKICAgICIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5zPTEsIG51bV9jbGFzc2VzPTIsCiAgICAgICAgICAgICAgICAgZW5jX2NoYW5uZWxzPU5vbmUsIGRfc3RhdGU9MTYsIGRfY29udj00LCBleHBhbmQ9Mik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgZW5jX2NoYW5uZWxzIGlzIE5vbmU6CiAgICAgICAgICAgIGVuY19jaGFubmVscyA9IFszMiwgNjQsIDEyOCwgMjU2XQoKICAgICAgICBib3R0bGVuZWNrX2NoID0gZW5jX2NoYW5uZWxzWy0xXSAqIDIgICAjIDUxMgoKICAgICAgICAjIC0tLS0gRW5jb2RlciAtLS0tCiAgICAgICAgc2VsZi5lbmNfc3RhZ2VzID0gbm4uTW9kdWxlTGlzdCgpCiAgICAgICAgaW5fY2ggPSBpbl9jaGFucwogICAgICAgIGZvciBvdXRfY2ggaW4gZW5jX2NoYW5uZWxzOgogICAgICAgICAgICBzZWxmLmVuY19zdGFnZXMuYXBwZW5kKAogICAgICAgICAgICAgICAgRW5jb2RlclN0YWdlKGluX2NoLCBvdXRfY2gsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZF9zdGF0ZT1kX3N0YXRlLCBkX2NvbnY9ZF9jb252LCBleHBhbmQ9ZXhwYW5kKQogICAgICAgICAgICApCiAgICAgICAgICAgIGluX2NoID0gb3V0X2NoCiAgICAgICAgc2VsZi5wb29sID0gbm4uTWF4UG9vbDJkKDIpCgogICAgICAgICMgLS0tLSBCb3R0bGVuZWNrIC0tLS0KICAgICAgICBzZWxmLmJvdHRsZW5lY2tfcmVzICAgPSBSZXNDb252QmxvY2soZW5jX2NoYW5uZWxzWy0xXSwgYm90dGxlbmVja19jaCkKICAgICAgICBzZWxmLmJvdHRsZW5lY2tfbWFtYmEgPSBNYW1iYUxheWVyKGJvdHRsZW5lY2tfY2gsIGRfc3RhdGU9ZF9zdGF0ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRfY29udj1kX2NvbnYsIGV4cGFuZD1leHBhbmQpCgogICAgICAgICMgLS0tLSBEZWNvZGVyIC0tLS0KICAgICAgICAjIENoYW5uZWxzOiBib3R0bGVuZWNr4oaSZW5jWy0xXSwgZW5jWy0xXeKGkmVuY1stMl0sIC4uLiwgZW5jWzFd4oaSZW5jWzBdCiAgICAgICAgc2VsZi5kZWNfc3RhZ2VzID0gbm4uTW9kdWxlTGlzdCgpCiAgICAgICAgZGVjX2luID0gYm90dGxlbmVja19jaAogICAgICAgIGZvciBza2lwX2NoIGluIHJldmVyc2VkKGVuY19jaGFubmVscyk6CiAgICAgICAgICAgIHNlbGYuZGVjX3N0YWdlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICBEZWNvZGVyU3RhZ2UodXBfY2g9ZGVjX2luLCBza2lwX2NoPXNraXBfY2gsIG91dF9jaD1za2lwX2NoKQogICAgICAgICAgICApCiAgICAgICAgICAgIGRlY19pbiA9IHNraXBfY2gKCiAgICAgICAgIyAtLS0tIEhlYWQgLS0tLQogICAgICAgIHNlbGYuaGVhZCA9IG5uLkNvbnYyZChlbmNfY2hhbm5lbHNbMF0sIG51bV9jbGFzc2VzLCAxKQoKICAgICAgICBzZWxmLl9pbml0X3dlaWdodHMoKQoKICAgIGRlZiBfaW5pdF93ZWlnaHRzKHNlbGYpOgogICAgICAgIGZvciBtIGluIHNlbGYubW9kdWxlcygpOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKG0sIChubi5Db252MmQsIG5uLkNvbnZUcmFuc3Bvc2UyZCkpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5rYWltaW5nX25vcm1hbF8obS53ZWlnaHQsIG1vZGU9J2Zhbl9vdXQnLCBub25saW5lYXJpdHk9J3JlbHUnKQogICAgICAgICAgICAgICAgaWYgbS5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG0uYmlhcykKICAgICAgICAgICAgZWxpZiBpc2luc3RhbmNlKG0sIG5uLkJhdGNoTm9ybTJkKToKICAgICAgICAgICAgICAgIG5uLmluaXQub25lc18obS53ZWlnaHQpCiAgICAgICAgICAgICAgICBubi5pbml0Lnplcm9zXyhtLmJpYXMpCiAgICAgICAgICAgIGVsaWYgaXNpbnN0YW5jZShtLCBubi5MaW5lYXIpOgogICAgICAgICAgICAgICAgbm4uaW5pdC50cnVuY19ub3JtYWxfKG0ud2VpZ2h0LCBzdGQ9MC4wMikKICAgICAgICAgICAgICAgIGlmIG0uYmlhcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBubi5pbml0Lnplcm9zXyhtLmJpYXMpCiAgICAgICAgICAgIGVsaWYgaXNpbnN0YW5jZShtLCBubi5MYXllck5vcm0pOgogICAgICAgICAgICAgICAgbm4uaW5pdC5vbmVzXyhtLndlaWdodCkKICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG0uYmlhcykKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBza2lwcyA9IFtdCgogICAgICAgICMgRW5jb2RlcgogICAgICAgIGZvciBzdGFnZSBpbiBzZWxmLmVuY19zdGFnZXM6CiAgICAgICAgICAgIHggPSBzdGFnZSh4KQogICAgICAgICAgICBza2lwcy5hcHBlbmQoeCkKICAgICAgICAgICAgeCA9IHNlbGYucG9vbCh4KQoKICAgICAgICAjIEJvdHRsZW5lY2sKICAgICAgICB4ID0gc2VsZi5ib3R0bGVuZWNrX3Jlcyh4KQogICAgICAgIHggPSBzZWxmLmJvdHRsZW5lY2tfbWFtYmEoeCkKCiAgICAgICAgIyBEZWNvZGVyCiAgICAgICAgZm9yIGRlY19zdGFnZSwgc2tpcCBpbiB6aXAoc2VsZi5kZWNfc3RhZ2VzLCByZXZlcnNlZChza2lwcykpOgogICAgICAgICAgICB4ID0gZGVjX3N0YWdlKHgsIHNraXApCgogICAgICAgICMgSGVhZAogICAgICAgIHggPSBzZWxmLmhlYWQoeCkKICAgICAgICByZXR1cm4geAoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUXVpY2sgc2FuaXR5IGNoZWNrCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgIyBQYXJhbWV0ZXIgY291bnQgV0lUSE9VVCBydW5uaW5nIG1hbWJhIChyZXBsYWNlIE1hbWJhTGF5ZXIgd2l0aCBpZGVudGl0eS1saWtlIHN0dWIpCiAgICBjbGFzcyBfRmFrZU1hbWJhKG5uLk1vZHVsZSk6CiAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWwsICoqa3dhcmdzKToKICAgICAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgICAgIHNlbGYucHJvaiA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsKQogICAgICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgICAgICByZXR1cm4gc2VsZi5wcm9qKHgpCgogICAgaW1wb3J0IHVuaXR0ZXN0Lm1vY2sgYXMgbW9jawoKICAgIGZha2VfbWFtYmFfbW9kdWxlID0gbW9jay5NYWdpY01vY2soKQogICAgZmFrZV9tYW1iYV9tb2R1bGUuTWFtYmEgPSBfRmFrZU1hbWJhCgogICAgaW1wb3J0IHN5cwogICAgc3lzLm1vZHVsZXNbJ21hbWJhX3NzbSddID0gZmFrZV9tYW1iYV9tb2R1bGUKCiAgICBtb2RlbCA9IFVNYW1iYShpbl9jaGFucz0xLCBudW1fY2xhc3Nlcz0yKQogICAgbl9wYXJhbXMgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHByaW50KGYnVS1NYW1iYSAgcGFyYW1ldGVyczoge25fcGFyYW1zIC8gMWU2Oi4xZn0gTScpCiAgICB4ID0gdG9yY2gucmFuZG4oMiwgMSwgMjU2LCAyNTYpCiAgICB5ID0gbW9kZWwoeCkKICAgIHByaW50KGYnSW5wdXQ6IHt4LnNoYXBlfSAg4oaSICBPdXRwdXQ6IHt5LnNoYXBlfScpCiAgICBhc3NlcnQgeS5zaGFwZSA9PSAoMiwgMiwgMjU2LCAyNTYpLCBmJ1VuZXhwZWN0ZWQgb3V0cHV0IHNoYXBlOiB7eS5zaGFwZX0nCiAgICBwcmludCgnT0snKQo="
with open("models/umamba.py", "wb") as _f:
    _f.write(base64.b64decode(_uma_b64))

print("models/ updated: segformer + swin_unet + umamba ditambahkan.")


In [ ]:
import os, csv, json, math, time, random, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
from timm.models.layers import trunc_normal_
from scipy.ndimage import distance_transform_edt
from sklearn.model_selection import KFold

try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print("mamba_ssm loaded successfully.")
except Exception as e:
    MAMBA_AVAILABLE = False
    print("WARNING: mamba_ssm not available — vm_unet_baseline tidak bisa dijalankan (OK untuk sesi 1 & 2).")

try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except Exception:
    THOP_AVAILABLE = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

AMP_DEVICE_TYPE = "cuda" if device.type == "cuda" else "cpu"
USE_AMP    = (device.type == "cuda")
PIN_MEMORY = (device.type == "cuda")

print(f"Device  : {device}")
print(f"AMP     : {USE_AMP} ({AMP_DEVICE_TYPE})")
print(f"THOP    : {THOP_AVAILABLE}")
print(f"Torch   : {torch.__version__}")


import sys
sys.path.insert(0, './models')
print('models path set')

In [ ]:
# ============================================================
# JALUR B — melengkapi roster ke k = 10 arsitektur
# ============================================================
# MENGAPA. Tesis paper ("Dice adalah proxy peringkat yang lemah") diuji dengan
# Spearman rho antara peringkat-Dice dan peringkat-CIMT lintas arsitektur.
# Pada k = 6 sekarang: rho = +0.20, TAPI nilai kritis (alfa .05) adalah 0.886 —
# uji itu tidak punya daya sama sekali. Pada k = 10 nilai kritisnya turun ke
# ~0.648, dan untuk pertama kalinya hasilnya bisa berarti.
#
# Sudah selesai 5-fold (7): unet, attention_unet, segformer, swin_unet, unext,
#                           umamba (2 run), + BC-VMamba (usulan).
# Yang kurang (3)         : unetpp, resunet, transunet   <- notebook ini.
#
# CARA PAKAI — SATU MODEL PER SESI KAGGLE (tiap model 5-fold ~5-6 jam):
#   sesi 1: BASELINE_NAMES = ["unetpp"]
#   sesi 2: BASELINE_NAMES = ["resunet"]
#   sesi 3: BASELINE_NAMES = ["transunet"]
#
# WAJIB: protokol, fold, dan SEED TIDAK BOLEH diubah. Keabsahan uji rho
# bergantung sepenuhnya pada semua arsitektur dilatih di bawah kondisi identik.
#
# PASTIKAN `{model}_fold_metrics.csv` IKUT TERSIMPAN di output — dua run parsial
# sebelumnya (unetpp 1 fold, resunet 2 fold) gagal justru di situ, dan tanpa
# berkas itu R2_baseline_ranking.py tidak bisa membaca hasilnya.
#
# Setelah ketiganya turun -> unduh ke:
#   Experiment baru 2/hasil run kaggle/baseline comparison/results/baselines/{model}/
# lalu jalankan lokal:  python notebooks/R2_baseline_ranking.py
# ============================================================
BASELINE_NAMES = ["unetpp"]   # <<< GANTI TIAP SESI: unetpp -> resunet -> transunet

IMG_SIZE       = 256     # PDF: standard CUBS preprocessing 256x256
K_FOLDS        = 5       # final 5-fold patient-disjoint (E0)
SEED           = 42
BATCH_SIZE     = 8       # PDF: 8 (T4x2) / 16 (A100)
NUM_WORKERS    = 2
NUM_EPOCHS     = 100     # max epoch untuk cek batas konvergensi
WARMUP_EPOCHS  = 5       # linear warmup 1e-2*LR -> LR selama 5 ep
LR             = 1e-4    # PDF: AdamW lr=1e-4
WEIGHT_DECAY   = 1e-4    # PDF: weight_decay=1e-4
ETA_MIN        = 1e-6    # Cosine Annealing eta_min
EARLY_STOP_PAT = 15      # PDF: early stop patience=15

# ============================================================
# Environment & Path Detection (Kaggle / Colab / Local)
# ============================================================
import os
from pathlib import Path

_ON_KAGGLE = Path('/kaggle/input').exists()
_ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

if _ON_COLAB:
    # Mount Google Drive (muncul popup izin -- klik Connect)
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')

    _DRIVE     = Path('/content/drive/MyDrive')
    _CUBS_BASE = _DRIVE / "CUBS"     # <- ubah jika nama folder berbeda

    def _find(pat):
        hits = sorted(_CUBS_BASE.glob(pat))
        if hits: return hits[0]
        for _s in sorted(_CUBS_BASE.iterdir()) if _CUBS_BASE.exists() else []:
            if _s.is_dir():
                hits = sorted(_s.glob(pat))
                if hits: return hits[0]
        return None

    _cubs21_dir = (_find("DATASET for Carotid*") or _find("DATASET_CUBS_clin*")
                   or _CUBS_BASE / "DATASET for Carotid Ultrasound Boundary Study CUBS an open multi-center analysis of computerized intima-media thickness measurement systems and their clinical impact")
    _cubs22_dir = _find("DATASET_CUBS_tech") or _CUBS_BASE / "DATASET_CUBS_tech"
    _split_path = _find("split_info.json")   or _CUBS_BASE / "split_info.json"
    OUTPUT_BASE = _DRIVE / "bcvmamba_baseline_results/baselines"

    print(f"Colab Drive paths (CUBS_BASE={_CUBS_BASE}):")
    _ok_all = True
    for _lbl, _p in [("cubs21", _cubs21_dir), ("cubs22", _cubs22_dir), ("split", _split_path)]:
        _ok = _p is not None and Path(str(_p)).exists()
        _ok_all = _ok_all and _ok
        print(f"  {'OK  ' if _ok else 'MISS'}: {_lbl}: {_p}")
    if not _ok_all:
        print(f"\n  Isi folder CUBS di Drive:")
        try:
            for _it in sorted(_CUBS_BASE.iterdir()):
                print(f"    {'[D]' if _it.is_dir() else '[F]'} {_it.name}")
        except Exception as _ex:
            print(f"    (error: {_ex})")
        print("  Ubah _CUBS_BASE di atas jika nama folder beda, lalu re-run cell ini.")

elif _ON_KAGGLE:
    def _kfind(pat):
        hits = sorted(Path('/kaggle/input').rglob(pat))
        return hits[0] if hits else None
    _cubs21_dir = _kfind("DATASET for Carotid*")
    _cubs22_dir = _kfind("DATASET_CUBS_tech")
    _split_path = _kfind("split_info.json")
    OUTPUT_BASE = Path("./results/baselines")
    print(f"Kaggle paths: cubs21={_cubs21_dir}, cubs22={_cubs22_dir}")

else:
    _LOCAL      = Path("../dataset")
    _cubs21_dir = _LOCAL / "DATASET for Carotid Ultrasound Boundary Study CUBS an open multi-center analysis of computerized intima-media thickness measurement systems and their clinical impact"
    _cubs22_dir = _LOCAL / "DATASET/CUBStech"
    _split_path = _LOCAL / "outputs/split_info.json"
    OUTPUT_BASE = Path("./results/baselines")
    print("Local mode")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
KAGGLE_INPUT = Path('/kaggle/input')  # selalu didefinisikan; cells berikutnya pakai .exists() check

print(f"\nBaselines  : {BASELINE_NAMES}")
print(f"Epochs     : {NUM_EPOCHS}  (warmup={WARMUP_EPOCHS}, early_stop_pat={EARLY_STOP_PAT})")
print(f"K_FOLDS    : {K_FOLDS}  ({'convergence check 80/20 split' if K_FOLDS==1 else 'final CV'})")
print(f"LR / Batch : {LR} / {BATCH_SIZE}  (AdamW, weight_decay={WEIGHT_DECAY})")
print(f"Output     : {OUTPUT_BASE.resolve()}")
for _lbl, _p in [("cubs21", _cubs21_dir), ("cubs22", _cubs22_dir), ("split", _split_path)]:
    _st = "OK" if _p is not None and Path(str(_p)).exists() else "NOT FOUND"
    print(f"  {_st}  {_lbl}: {_p}")


In [ ]:
"""
E1a — DATA LOADING PENGGANTI  (drop-in untuk CELL 5 + CELL 6 notebook lama)
===========================================================================
Ganti isi CELL 5 dan CELL 6 di `bcvmamba_cubs_a1.ipynb` (atau notebook baseline)
dengan file ini. SEMUA cell lain (model, loss, training driver, plotting) TIDAK berubah.

PERBEDAAN vs data loading lama:
  - GT mask = anotator A1 (GOLD STANDARD), DIRASTERISASI on-the-fly dari profil LI/MA.
    (Lama: memuat `masks/{id}_mask.png` yang ternyata = A1', bukan A1.)
  - ROI crop = dari rect A1 (`Manual-A1/{id}_rect.txt`). (Lama: dari annotation_log = A1'.)
  - Split = held-out test (401) + 5-fold CV PATIENT-DISJOINT dari E0 (`splits_5fold.json`).
    (Lama: KFold pada nama citra -> ada bocor L/R.)
  - Metadata (center/SNR/morph) dari `master_index.csv` (E0).

>>> SATU-SATUNYA ubahan di luar cell ini <<<
    Di CELL 11 (run_baseline_training), ganti baris:
        fold_splits = list(KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED).split(cv_names))
    menjadi:
        fold_splits = PATIENT_FOLD_SPLITS        # 5-fold patient-disjoint dari E0
    (baris K_FOLDS==1 boleh dibiarkan; untuk K_FOLDS==5 dipakai PATIENT_FOLD_SPLITS.)

>>> UPLOAD ke Kaggle <<<
    Sertakan 2 file kecil dari E0 sebagai Kaggle dataset (atau taruh di working dir):
        master_index.csv , splits_5fold.json
    Skrip akan mencarinya otomatis di /kaggle/input (rglob) atau ./ .

Mengandalkan variabel dari CELL 4: CUBS21_DIR (_cubs21_dir), CUBS22_DIR (_cubs22_dir),
IMG_SIZE, BATCH_SIZE, NUM_WORKERS, PIN_MEMORY, SEED, KAGGLE_INPUT.
"""
# ================================================================= CELL 5 (pengganti)
import json, csv, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import cv2

GT_ANNOT = "Manual-A1"    # gold standard

CUBS21_DIR = Path(str(_cubs21_dir))
CUBS22_DIR = Path(str(_cubs22_dir))

def _find_file(fname):
    """Cari file (master_index.csv / splits_5fold.json) di Kaggle input atau cwd."""
    for base in [Path("."), Path("./data"), KAGGLE_INPUT if 'KAGGLE_INPUT' in globals() else Path("/kaggle/input")]:
        hit = list(Path(base).rglob(fname)) if Path(base).exists() else []
        if hit:
            return hit[0]
    raise FileNotFoundError(f"{fname} tidak ditemukan. Upload output E0 ke Kaggle.")

MASTER_CSV  = _find_file("master_index.csv")
SPLITS_JSON = _find_file("splits_5fold.json")
print(f"master_index: {MASTER_CSV}")
print(f"splits      : {SPLITS_JSON}")

master = pd.read_csv(MASTER_CSV)
META   = master.set_index("image_id").to_dict("index")   # id -> {center,snr,morph,release,...}

@dataclass
class FoldResult:
    fold: int; dice: float; iou: float; accuracy: float; recall: float
    precision: float; mae_cimt_mm: float; mse_cimt_mm: float
    params_m: float; gflops: float; latency_ms: float; max_mem_mb: float
    best_epoch: int = 0

# ---- resolusi path per image_id (portable: dari CUBS21_DIR/CUBS22_DIR, bukan path absolut CSV)
def resolve_paths(iid: str):
    if iid.startswith("clin_"):
        seg = CUBS21_DIR / "SEGMENTATIONS" / GT_ANNOT
        return dict(img=CUBS21_DIR/"IMAGES"/f"{iid}.tiff",
                    li=seg/f"{iid}-LI.txt", ma=seg/f"{iid}-MA.txt",
                    rect=seg/f"{iid}_rect.txt", cf=CUBS21_DIR/"CF"/f"{iid}_CF.txt")
    seg = CUBS22_DIR / "LIMA-Profiles" / GT_ANNOT
    return dict(img=CUBS22_DIR/"images"/f"{iid}.tiff",
                li=seg/f"{iid}-LI.txt", ma=seg/f"{iid}-MA.txt",
                rect=seg/f"{iid}_rect.txt", cf=CUBS22_DIR/"CF"/f"{iid}_CF.txt")

def load_calibration_factor(iid: str):
    try:
        return float(Path(resolve_paths(iid)["cf"]).read_text().strip().split()[0])
    except Exception:
        return None

# ---- split E0 -> cv_names, test_names, PATIENT_FOLD_SPLITS (patient-disjoint)
_split = json.load(open(SPLITS_JSON, encoding="utf-8"))
_pid2imgs = master.groupby("patient_id")["image_id"].apply(list).to_dict()

test_names = np.array([im for p in _split["test"] for im in _pid2imgs[p]])
_cv_list, _fold_tag = [], []
for k in [f"fold{i+1}" for i in range(5)]:
    for p in _split["folds"][k]:
        for im in _pid2imgs[p]:
            _cv_list.append(im); _fold_tag.append(k)
cv_names   = np.array(_cv_list)
_fold_tag  = np.array(_fold_tag)
PATIENT_FOLD_SPLITS = [(np.where(_fold_tag != k)[0], np.where(_fold_tag == k)[0])
                       for k in [f"fold{i+1}" for i in range(5)]]

print(f"cv_names   : {len(cv_names)}  (5-fold patient-disjoint)")
print(f"test_names : {len(test_names)}  (held-out)")
print(f"folds (val): {[len(v) for _, v in PATIENT_FOLD_SPLITS]}")



In [ ]:
# ================================================================= CELL 6 (pengganti)
import random, torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

def load_profile(txt):
    pts = []
    for line in Path(txt).read_text().splitlines():
        s = line.split()
        if len(s) >= 2:
            pts.append((float(s[0]), float(s[1])))
    return np.asarray(pts, dtype=np.float32)

def rasterize_imc(li, ma, hw):
    """Mask IMC = area antara batas LI (atas) & MA (bawah)."""
    H, W = hw
    if len(li) < 2 or len(ma) < 2:
        return np.zeros((H, W), np.uint8)
    li = li[np.argsort(li[:, 0])]
    ma = ma[np.argsort(ma[:, 0])][::-1]
    poly = np.vstack([li, ma]).astype(np.int32)
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [poly], 1)
    return m

def read_rect(rect_path):
    try:
        v = Path(rect_path).read_text().split()
        return float(v[0]), float(v[1]), float(v[2]), float(v[3])   # x y w h
    except Exception:
        return None

def crop_roi_with_padding(image, roi_x, roi_y, roi_w, roi_h, v_pad=1.0, h_pad=0.1):
    h, w = image.shape[:2]
    pad_v = roi_h * v_pad
    y0 = int(max(0, roi_y - pad_v / 2)); y1 = int(min(h, roi_y + roi_h + pad_v / 2))
    pad_h = roi_w * h_pad
    x0 = int(max(0, roi_x - pad_h));     x1 = int(min(w, roi_x + roi_w + pad_h))
    cropped = image[y0:y1, x0:x1]
    ch, cw = cropped.shape[:2]
    sq = max(ch, cw, 1)
    square = np.zeros((sq, sq), dtype=image.dtype)
    yo, xo = (sq - ch) // 2, (sq - cw) // 2
    square[yo:yo+ch, xo:xo+cw] = cropped
    return square, float(sq)

def compute_dataset_stats(names, image_size=256):
    s, sq, n = 0.0, 0.0, 0
    for name in tqdm(names, desc="mean/std", leave=False):
        p = resolve_paths(str(name))
        img = cv2.imread(str(p["img"]), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        rect = read_rect(p["rect"])
        if rect is not None:
            img, _ = crop_roi_with_padding(img, *rect)
        img = cv2.resize(img, (image_size, image_size))
        pix = img.astype(np.float64) / 255.0
        nz = pix > 0
        if nz.any():
            c = pix[nz]; s += c.sum(); sq += (c**2).sum(); n += c.size
    mean = s / max(1, n)
    return float(mean), float(max(np.sqrt(sq / max(1, n) - mean**2), 1e-6))

DATASET_MEAN, DATASET_STD = compute_dataset_stats(cv_names, IMG_SIZE)
print(f"DATASET_MEAN={DATASET_MEAN:.6f}, DATASET_STD={DATASET_STD:.6f}")

def _acoustic_shadow(img, factor=0.5):
    w = img.shape[1]; sw = max(1, random.randint(w//8, w//4)); sx = random.randint(0, max(0, w-sw))
    out = img.astype(np.float32).copy(); out[:, sx:sx+sw] = np.clip(out[:, sx:sx+sw]*factor, 0, 255)
    return out.astype(img.dtype)
def _acoustic_enh(img, factor=1.5):
    h = img.shape[0]; eh = max(1, random.randint(h//8, h//4)); ey = random.randint(0, max(0, h-eh))
    out = img.astype(np.float32).copy(); out[ey:ey+eh, :] = np.clip(out[ey:ey+eh, :]*factor, 0, 255)
    return out.astype(img.dtype)

def get_train_transform():
    return A.Compose([
        A.Affine(translate_percent={"x": (-0.2, 0.2), "y": (-0.2, 0.2)}, scale=(0.8, 1.2),
                 rotate=(-30, 30), shear=(-30, 30), border_mode=cv2.BORDER_CONSTANT, p=0.7),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomResizedCrop(size=(256, 256), scale=(0.25, 1.0), ratio=(1.0, 1.0), p=0.3),
        A.ElasticTransform(alpha=10, sigma=5, border_mode=cv2.BORDER_CONSTANT, p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.2, border_mode=cv2.BORDER_CONSTANT, p=0.3),
        A.Lambda(image=lambda img, **kw: _acoustic_shadow(img, 0.5), p=0.5),
        A.Lambda(image=lambda img, **kw: _acoustic_enh(img, 1.5), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.5, contrast_limit=0.5, p=0.5),
        A.GaussNoise(std_range=(0.05, 0.1), mean_range=(0.0, 0.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 3), p=0.3), A.MedianBlur(blur_limit=3, p=0.3),
        A.MotionBlur(blur_limit=3, p=0.3), A.Equalize(p=0.3),
        A.Normalize(mean=(DATASET_MEAN,), std=(DATASET_STD,), max_pixel_value=255.0), ToTensorV2()])

def get_val_transform():
    return A.Compose([A.Normalize(mean=(DATASET_MEAN,), std=(DATASET_STD,), max_pixel_value=255.0),
                      ToTensorV2()])

def measure_imt_from_mask(binary_mask, calibration_factor, roi_square_side, img_size=256):
    """Column-wise mean CIMT (mm) — sama seperti versi lama (per kolom LI->MA)."""
    scale = roi_square_side / float(img_size)
    b = (binary_mask > 0.5)
    th = []
    for col in range(b.shape[1]):
        rows = np.where(b[:, col])[0]
        if len(rows) >= 2:
            th.append((rows[-1] - rows[0]) * scale * calibration_factor)
    return float(np.mean(th)) if th else 0.0

class CUBSCombinedDataset(Dataset):
    def __init__(self, names, image_size=256, train=True, mosaic_p=0.3):
        self.names = [str(n) for n in names]
        self.image_size = image_size
        self.train = train
        self.mosaic_p = mosaic_p if train else 0.0
        self.transform = get_train_transform() if train else get_val_transform()

    def _load_raw(self, name):
        p = resolve_paths(name)
        image = cv2.imread(str(p["img"]), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Image not found: {p['img']}")
        li, ma = load_profile(p["li"]), load_profile(p["ma"])
        mask = rasterize_imc(li, ma, image.shape) * 255      # 0/255
        rect = read_rect(p["rect"])
        if rect is not None:
            image, sq = crop_roi_with_padding(image, *rect)
            mask,  _  = crop_roi_with_padding(mask,  *rect)
        else:
            h, w = image.shape[:2]; sq = float(max(h, w))
        return image, mask, sq

    def _make_mosaic(self, idx):
        half = self.image_size // 2
        idxs = [idx] + random.choices(range(len(self.names)), k=3)
        imgs, masks = [], []
        for i in idxs:
            img, msk, _ = self._load_raw(self.names[i])
            imgs.append(cv2.resize(img, (half, half)))
            masks.append(cv2.resize(msk, (half, half), interpolation=cv2.INTER_NEAREST))
        mi = np.concatenate([np.concatenate([imgs[0], imgs[1]], 1),
                             np.concatenate([imgs[2], imgs[3]], 1)], 0)
        mm = np.concatenate([np.concatenate([masks[0], masks[1]], 1),
                             np.concatenate([masks[2], masks[3]], 1)], 0)
        return mi, mm

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        if self.train and random.random() < self.mosaic_p:
            image, mask = self._make_mosaic(idx); sq = float(self.image_size)
        else:
            image, mask, sq = self._load_raw(name)
            image = cv2.resize(image, (self.image_size, self.image_size))
            mask  = cv2.resize(mask,  (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)
        aug = self.transform(image=image, mask=mask)
        cf = load_calibration_factor(name)
        cimt_gt = measure_imt_from_mask(aug["mask"].numpy().astype(np.float32),
                                        cf, float(sq), self.image_size) if cf else 0.0
        return {"image": aug["image"], "mask": aug["mask"].long(), "name": name,
                "roi_square_side": sq, "cimt_gt": torch.tensor(cimt_gt, dtype=torch.float32)}

def make_dataloaders(train_names, val_names):
    persistent = NUM_WORKERS > 0
    tr = DataLoader(CUBSCombinedDataset(train_names, IMG_SIZE, True),
                    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, persistent_workers=persistent, drop_last=True)
    va = DataLoader(CUBSCombinedDataset(val_names, IMG_SIZE, False),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, persistent_workers=persistent)
    return tr, va

print("Dataset (A1-rasterized, patient-disjoint folds) ready.")


In [ ]:
from models import get_model, AVAILABLE

print("Sanity check semua baseline dalam BASELINE_NAMES:")
for _name in BASELINE_NAMES:
    m = get_model(_name, num_classes=2).to(device)
    m.eval()
    with torch.no_grad():
        dummy = torch.randn(1, 1, IMG_SIZE, IMG_SIZE, device=device)
        out   = m(dummy)
    assert out.shape == (1, 2, IMG_SIZE, IMG_SIZE), f"Wrong shape {_name}: {out.shape}"
    p = sum(x.numel() for x in m.parameters()) / 1e6
    print(f"  {_name:<22}: {p:.2f}M params — output {out.shape}  OK")
    del m, dummy, out
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Semua model OK.")


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1).clamp(1e-6, 1 - 1e-6)
        nc    = probs.shape[1]
        t_oh  = F.one_hot(targets.long(), nc).permute(0, 3, 1, 2).float()
        p_f   = probs.flatten(2); t_f = t_oh.flatten(2)
        intersection = (p_f * t_f).sum(-1)
        dice_idx = (2 * intersection + self.smooth) / (p_f.sum(-1) + t_f.sum(-1) + self.smooth)
        return 1 - dice_idx.mean()



def segmentation_metrics(logits, targets, eps=1e-6):
    preds   = torch.argmax(logits, dim=1)
    pred_fg = (preds == 1); true_fg = (targets == 1)
    tp = (pred_fg & true_fg).sum().float()
    fp = (pred_fg & ~true_fg).sum().float()
    fn = (~pred_fg & true_fg).sum().float()
    tn = (~pred_fg & ~true_fg).sum().float()
    return {
        "dice":      float((2*tp+eps)/(2*tp+fp+fn+eps)),
        "iou":       float((tp+eps)/(tp+fp+fn+eps)),
        "accuracy":  float((tp+tn+eps)/(tp+tn+fp+fn+eps)),
        "recall":    float((tp+eps)/(tp+fn+eps)),
        "precision": float((tp+eps)/(tp+fp+eps)),
    }



class SimpleLoss(nn.Module):
    """0.5 × DiceLoss + 0.5 × CrossEntropyLoss(weight=[1,3]) — isolasi arsitektur saja."""
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        ce_w = torch.tensor([1.0, 3.0], device=device)
        self.ce  = nn.CrossEntropyLoss(weight=ce_w)

    def forward(self, logits, targets):
        return 0.5 * self.dice(logits, targets) + 0.5 * self.ce(logits, targets.long())

print("SimpleLoss, DiceLoss, segmentation_metrics ready.")


In [ ]:
def measure_efficiency(model, img_size=256):
    model.eval()
    params_m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    gflops   = float("nan")
    if THOP_AVAILABLE:
        try:
            dummy   = torch.randn(1, 1, img_size, img_size, device=device)
            macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
            gflops  = float(2.0 * macs / 1e9)
        except Exception as e:
            warnings.warn(f"GFLOPs gagal: {e}")
    runs  = 30; warmup = 5
    dummy = torch.randn(1, 1, img_size, img_size, device=device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        for _ in range(warmup): _ = model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(runs): _ = model(dummy)
        e.record(); torch.cuda.synchronize()
        latency_ms = s.elapsed_time(e) / runs
        max_mem_mb = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        for _ in range(warmup): _ = model(dummy)
        t0 = time.perf_counter()
        for _ in range(runs): _ = model(dummy)
        latency_ms = ((time.perf_counter()-t0)/runs)*1000
        max_mem_mb = float("nan")
    return {"params_m": float(params_m), "gflops": float(gflops),
            "latency_ms": float(latency_ms), "max_mem_mb": float(max_mem_mb)}


print("measure_efficiency ready.")

In [ ]:
def train_one_epoch_baseline(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, n = 0.0, 0
    agg = {k: 0.0 for k in ["dice","iou","accuracy","recall","precision"]}
    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out  = model(imgs)
            logits = out[0] if isinstance(out, tuple) else out
            loss = criterion(logits, masks)
        if math.isnan(loss.item()) or math.isinf(loss.item()):
            optimizer.zero_grad(set_to_none=True); continue
        if USE_AMP and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        with torch.no_grad():
            m = segmentation_metrics(logits.detach(), masks)
        for k in agg: agg[k] += m[k] * bs
        n += bs
    return total_loss / max(1, n), {k: agg[k] / max(1, n) for k in agg}


@torch.no_grad()
def evaluate_model_baseline(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    agg = {k: 0.0 for k in ["dice","iou","accuracy","recall","precision"]}
    imt_abs_err = []
    for batch in loader:
        imgs   = batch["image"].to(device, non_blocking=True)
        masks  = batch["mask"].to(device, non_blocking=True)
        names  = batch["name"]
        roi_sq = batch["roi_square_side"]
        with torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out    = model(imgs)
            logits = out[0] if isinstance(out, tuple) else out
            loss   = criterion(logits, masks)
        bs = imgs.size(0)
        total_loss += loss.item() * bs
        m = segmentation_metrics(logits, masks)
        for k in agg: agg[k] += m[k] * bs
        preds = torch.argmax(logits, dim=1).cpu().numpy().astype(np.float32)
        gts   = masks.cpu().numpy().astype(np.float32)
        for i in range(bs):
            cf = load_calibration_factor(names[i])
            if cf is None: continue
            imt_pred = measure_imt_from_mask(preds[i], cf, float(roi_sq[i]), IMG_SIZE)
            imt_gt   = measure_imt_from_mask(gts[i],   cf, float(roi_sq[i]), IMG_SIZE)
            imt_abs_err.append(abs(imt_pred - imt_gt))
        n += bs
    mae = float(np.mean(imt_abs_err)) if imt_abs_err else float("nan")
    mse = float(np.mean([e**2 for e in imt_abs_err])) if imt_abs_err else float("nan")
    return total_loss / max(1, n), {k: agg[k] / max(1, n) for k in agg}, mae, mse

print("train_one_epoch_baseline / evaluate_model_baseline ready.")


In [ ]:
# ================================================================
# ADD-ON: simpan prediksi per-citra (CSV-only) untuk E8/E9 baseline
# val_fold{f}_per_image.csv & test_fold{f}_per_image.csv per model.
# (probs.npz TIDAK disimpan untuk baseline -> hemat storage; hanya perlu utk model usulan)
# ================================================================
import numpy as _np, pandas as _pd, torch as _torch
from torch.utils.data import DataLoader as _DL
_TEST_DL_CACHE = {}
def _get_test_loader():
    if "dl" not in _TEST_DL_CACHE:
        _ds = CUBSCombinedDataset(test_names, IMG_SIZE, train=False)
        _TEST_DL_CACHE["dl"] = _DL(_ds, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS>0))
    return _TEST_DL_CACHE["dl"]

@_torch.no_grad()
def _export_csv(model, loader, out_dir, tag):
    model.eval(); rows=[]
    for batch in loader:
        imgs  = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].numpy().astype(_np.float32)
        names = batch["name"]; roi_sq = batch["roi_square_side"]
        with _torch.amp.autocast(AMP_DEVICE_TYPE, enabled=USE_AMP):
            out = model(imgs); logits = out[0] if isinstance(out, tuple) else out
        pred = (_torch.softmax(logits.float(), 1)[:,1].cpu().numpy() > 0.5).astype(_np.float32)
        for i in range(len(names)):
            name=names[i]; p=pred[i]>0.5; g=masks[i]>0.5
            tp=float((p&g).sum()); fp=float((p&~g).sum()); fn=float((~p&g).sum()); eps=1e-6
            cf=load_calibration_factor(name)
            cp=measure_imt_from_mask(pred[i],  cf, float(roi_sq[i]), IMG_SIZE) if cf else _np.nan
            cg=measure_imt_from_mask(masks[i], cf, float(roi_sq[i]), IMG_SIZE) if cf else _np.nan
            m=META.get(name, {})
            rows.append(dict(name=name, release=m.get("release"), center=m.get("center"),
                snr=m.get("snr"), morph=m.get("morph"),
                dice=(2*tp+eps)/(2*tp+fp+fn+eps), iou=(tp+eps)/(tp+fp+fn+eps),
                recall=(tp+eps)/(tp+fn+eps), precision=(tp+eps)/(tp+fp+eps),
                cimt_pred_mm=cp, cimt_gt_mm=cg,
                imt_abs_err_mm=(abs(cp-cg) if cf else _np.nan),
                roi_square_side=float(roi_sq[i]), cf=cf))
    _pd.DataFrame(rows).to_csv(out_dir / f"{tag}_per_image.csv", index=False)
    return len(rows)

def save_fold_predictions(model, val_dl, fold, out_dir):
    nv=_export_csv(model, val_dl,            out_dir, f"val_fold{fold}")
    nt=_export_csv(model, _get_test_loader(), out_dir, f"test_fold{fold}")
    print(f"  [save preds] fold {fold}: val={nv} test={nt} -> {out_dir} (csv only)")

print("save_fold_predictions (CSV-only) ready.")


In [ ]:
def run_baseline_training():
    # ---- split: K_FOLDS=1 pakai 80/20 train_test_split; K_FOLDS>1 pakai KFold ----
    if K_FOLDS == 1:
        from sklearn.model_selection import train_test_split as _tts
        _tr, _va = _tts(np.arange(len(cv_names)), test_size=0.2, random_state=SEED)
        fold_splits = [(_tr, _va)]
        cv_label = "1-fold 80/20 split (convergence check)"
    else:
        fold_splits = PATIENT_FOLD_SPLITS        # 5-fold patient-disjoint dari E0
        cv_label = f"{K_FOLDS}-fold CV"

    fold_rows, history_rows = [], []

    print("=" * 70)
    print(f"Baseline Training: {BASELINE_NAME}")
    print(f"Epochs={NUM_EPOCHS} | Warmup={WARMUP_EPOCHS} | LR={LR} | Batch={BATCH_SIZE}")
    print(f"Loss: 0.5*Dice + 0.5*CE(w=[1,3])  |  Optimizer: AdamW")
    print(f"{cv_label} seed={SEED}  |  output: {OUTPUT_DIR}")
    print("=" * 70)

    for fold, (tr_idx, va_idx) in enumerate(fold_splits, 1):
        print(f"\n{'='*70}\nFOLD {fold}/{K_FOLDS}\n{'='*70}")

        train_dl, val_dl = make_dataloaders(cv_names[tr_idx], cv_names[va_idx])

        model = get_model(BASELINE_NAME, num_classes=2).to(device)
        eff   = measure_efficiency(model, IMG_SIZE)
        print(f"  Params={eff['params_m']:.3f}M | GFLOPs={eff['gflops']:.3f} | "
              f"Latency={eff['latency_ms']:.2f}ms | Mem={eff['max_mem_mb']:.1f}MB")

        criterion = SimpleLoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scaler    = torch.amp.GradScaler(AMP_DEVICE_TYPE, enabled=USE_AMP)

        warmup_sched = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=1e-2, end_factor=1.0, total_iters=WARMUP_EPOCHS)
        cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=ETA_MIN)
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])

        best_dice, best_state, patience_count, best_epoch = 0.0, None, 0, 1

        for epoch in range(1, NUM_EPOCHS + 1):
            tr_loss, tr_m = train_one_epoch_baseline(model, train_dl, optimizer, criterion, scaler)
            va_loss, va_m, va_mae, va_mse = evaluate_model_baseline(model, val_dl, criterion)
            scheduler.step()

            lr_now = optimizer.param_groups[0]["lr"]
            print(f"Fold {fold} | Ep {epoch:03d}/{NUM_EPOCHS} | lr={lr_now:.2e} | "
                  f"train={tr_loss:.4f} dice={tr_m['dice']:.4f} | "
                  f"val dice={va_m['dice']:.4f} iou={va_m['iou']:.4f} "
                  f"mae={va_mae:.4f}mm")

            history_rows.append({
                "fold": fold, "epoch": epoch, "lr": lr_now,
                "train_loss": tr_loss, "train_dice": tr_m["dice"],
                "val_loss": va_loss, "val_dice": va_m["dice"], "val_iou": va_m["iou"],
                "val_acc": va_m["accuracy"], "val_recall": va_m["recall"],
                "val_precision": va_m["precision"],
                "val_mae_cimt_mm": va_mae, "val_mse_cimt_mm": va_mse,
            })

            if va_m["dice"] > best_dice:
                best_dice   = va_m["dice"]
                best_state  = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                patience_count = 0
                best_epoch  = epoch
                torch.save({"fold": fold, "epoch": epoch, "best_val_dice": best_dice,
                            "model_state_dict": best_state, "baseline": BASELINE_NAME},
                           OUTPUT_DIR / f"{BASELINE_NAME}_fold{fold}_best.pth")
            else:
                patience_count += 1

            if patience_count >= EARLY_STOP_PAT:
                print(f"  [EarlyStop] ep={epoch} | patience={EARLY_STOP_PAT} exhausted | "
                      f"best_dice={best_dice:.4f} @ ep {best_epoch}")
                break

            if device.type == "cuda": torch.cuda.empty_cache()

        # Ringkasan konvergensi fold ini
        print(f"  => Konvergensi: best_dice={best_dice:.4f}  @ epoch {best_epoch} "
              f"(total run={epoch}/{NUM_EPOCHS})")

        if best_state is None:
            raise RuntimeError("No best_state saved.")
        model.load_state_dict(best_state)
        _, fold_m, fold_mae, fold_mse = evaluate_model_baseline(model, val_dl, criterion)
        save_fold_predictions(model, val_dl, fold, OUTPUT_DIR)   # simpan prediksi per-citra (E8/E9)
        torch.save({"fold": fold, "model_state_dict": model.state_dict(),
                    "best_val_dice": best_dice, "best_epoch": best_epoch,
                    "baseline": BASELINE_NAME},
                   OUTPUT_DIR / f"{BASELINE_NAME}_fold{fold}.pth")

        fold_rows.append(FoldResult(
            fold=fold, dice=fold_m["dice"], iou=fold_m["iou"],
            accuracy=fold_m["accuracy"], recall=fold_m["recall"],
            precision=fold_m["precision"], mae_cimt_mm=fold_mae, mse_cimt_mm=fold_mse,
            params_m=eff["params_m"], gflops=eff["gflops"],
            latency_ms=eff["latency_ms"], max_mem_mb=eff["max_mem_mb"],
            best_epoch=best_epoch,
        ))

    return pd.DataFrame([vars(r) for r in fold_rows]), pd.DataFrame(history_rows)

print("run_baseline_training() ready.")


In [ ]:
import gc, math as _math

all_fold_dfs = {}
all_hist_dfs = {}

metrics_order = ["dice","iou","accuracy","recall","precision",
                 "mae_cimt_mm","mse_cimt_mm","best_epoch",
                 "params_m","gflops","latency_ms","max_mem_mb"]
bcvmamba_ref = {"dice": 0.8491, "iou": 0.7385, "mae_cimt_mm": 0.1199, "params_m": 9.63}  # BC-VMamba +L_cimt (usulan, E1c)

for BASELINE_NAME in BASELINE_NAMES:
    OUTPUT_DIR = OUTPUT_BASE / BASELINE_NAME
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n{'#'*70}")
    print(f"  STARTING: {BASELINE_NAME}")
    print(f"{'#'*70}")

    fold_df, hist_df = run_baseline_training()
    all_fold_dfs[BASELINE_NAME] = fold_df
    all_hist_dfs[BASELINE_NAME] = hist_df

    fold_csv = OUTPUT_DIR / f"{BASELINE_NAME}_fold_metrics.csv"
    hist_csv = OUTPUT_DIR / f"{BASELINE_NAME}_training_history.csv"
    fold_df.to_csv(fold_csv, index=False)
    hist_df.to_csv(hist_csv, index=False)

    print("\n" + "=" * 70)
    print(f"SELESAI — {BASELINE_NAME.upper()}")
    print("=" * 70)
    n_folds = len(fold_df)
    w = 9
    hdr = f"  {'Metric':<22}" + "".join([f"  {'Fold '+str(i+1):>{w}}" for i in range(n_folds)])
    hdr += f"  {'Mean':>{w}}  {'Std':>{w}}"
    print(hdr); print("  " + "-" * (24 + (w+2) * (n_folds+2)))
    for m in metrics_order:
        if m in fold_df.columns:
            vals = fold_df[m].values
            row  = "".join([f"  {v:>{w}.4f}" for v in vals])
            row += f"  {vals.mean():>{w}.4f}  {vals.std():>{w}.4f}"
            print(f"  {m:<22}{row}")

    print(f"\n  Benchmark vs BC-VMamba +L_cimt (usulan, E1c):")
    for m in ["dice","iou","mae_cimt_mm","params_m"]:
        v = fold_df[m].mean()
        b = bcvmamba_ref.get(m, float("nan"))
        delta_s = f"{v-b:+.4f}" if not _math.isnan(b) else "  —"
        print(f"  {m:<22}: baseline={v:.4f}  BC-VMamba={b:.4f}  delta={delta_s}")
    print(f"  Artifacts: {fold_csv.parent.resolve()}")

    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*70}")
print(f"SESI SELESAI — {len(BASELINE_NAMES)} baseline: {BASELINE_NAMES}")
print(f"{'='*70}")


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if not all_hist_dfs:
    print("Jalankan training terlebih dahulu.")
else:
    panel_cfg = [
        ("val_dice",        "Val Dice"),
        ("val_loss",        "Val Loss"),
        ("val_mae_cimt_mm", "Val MAE CIMT (mm)"),
    ]
    for bname, hist_df in all_hist_dfs.items():
        out_dir = OUTPUT_BASE / bname
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, (col, title) in zip(axes, panel_cfg):
            for fid in sorted(hist_df["fold"].unique()):
                df_f = hist_df[hist_df["fold"] == fid]
                ax.plot(df_f["epoch"], df_f[col], label=f"Fold {fid}", linewidth=1.5)
            if col == "val_dice":
                ax.axhline(0.8491, color="steelblue", linestyle=":", alpha=0.8, linewidth=1.2, label="BC-VMamba +L_cimt ref")
            ax.set_title(f"{bname} — {title}")
            ax.set_xlabel("Epoch"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
        plt.suptitle(f"Baseline {bname} vs BC-VMamba | {NUM_EPOCHS}ep AdamW SimpleLoss", fontsize=10)
        plt.tight_layout()
        plot_path = out_dir / f"{bname}_training_curves.png"
        plt.savefig(plot_path, dpi=150); plt.show()
        print(f"Saved: {plot_path}")


## Urutan Eksekusi Kaggle (1 model per sesi Save Version)

Ubah `BASELINE_NAMES` di cell Config jadi **list satu-elemen**, lalu **Save Version**. Kaggle reset 30 jam/minggu; tiap model 5-fold ≈ 5–6 jam.

| Sesi | `BASELINE_NAMES` | Paradigma | Est. | Dalam quota 12h? |
|---|---|---|---|---|
| 1 | `["umamba"]` | Mamba (rival langsung) | ~6 jam | ✓ |
| 2 | `["swin_unet"]` | Transformer | ~6 jam | ✓ |
| 3 | `["attention_unet"]` | CNN + attention | ~5 jam | ✓ |

Model tambahan opsional: `unet, unetpp, resunet, transunet, unext, segformer, vm_unet_baseline`.

**Catatan**: Section analisis (Section 2) memuat **semua hasil yang sudah tersedia** — bisa dijalankan parsial setelah tiap sesi. Perbandingan definitif (uji signifikansi + CI) dikerjakan lokal di `E10_significance.py` setelah semua hasil terkumpul.

Referensi model usulan: **BC-VMamba +L_cimt (E1c)** — hasil di `hasil run kaggle/result e1c/.../bc_vmamba_cubs_lcimt_s42/`.


---
## Section 2 — Analisis Perbandingan

Cell-cell ini memuat **semua baseline yang sudah selesai** + BC-VMamba (+L_cimt, usulan) lalu menampilkan:
- Tabel mean ± std per model
- Uji statistik (paired t-test + Wilcoxon)
- Box plot distribusi Dice + Bar MAE + Scatter Params vs Dice

Bisa dijalankan ulang kapan saja — analisis otomatis parsial jika belum semua selesai.


In [ ]:
import math as _math
from scipy.stats import ttest_rel, wilcoxon

# Load semua fold_metrics.csv yang sudah ada
BASELINES_DIR = Path("./results/baselines")
PROPOSED_CSV  = Path("./results/baselines/bc_vmamba_cubs_lcimt_s42/bc_vmamba_cubs_a1_fold_metrics.csv")  # BC-VMamba +L_cimt (E1c); sesuaikan path saat analisis lokal

results = {}

if PROPOSED_CSV.exists():
    results["BC-VMamba +L_cimt (ours)"] = pd.read_csv(PROPOSED_CSV)
    print(f"Loaded BC-VMamba +L_cimt: {len(results['BC-VMamba +L_cimt (ours)'])} folds")
else:
    print(f"WARNING: BC-VMamba +L_cimt fold_metrics tidak ditemukan — perbandingan vs usulan dilakukan lokal (E10)")

baseline_labels = {
    "unet":             "U-Net",
    "unetpp":           "UNet++",
    "resunet":          "ResU-Net",
    "attention_unet":   "Attention U-Net",
    "transunet":        "TransUNet",
    "unext":            "UNeXt",
    "segformer":        "SegFormer",
    "swin_unet":        "Swin-UNet",
    "umamba":           "U-Mamba",
    "vm_unet_baseline": "VM-UNet baseline",
}
for name, label in baseline_labels.items():
    csv_path = BASELINES_DIR / name / f"{name}_fold_metrics.csv"
    if csv_path.exists():
        results[label] = pd.read_csv(csv_path)
        print(f"  Loaded {label}: {len(results[label])} folds")
    else:
        print(f"  (not yet) {label}")

print(f"\nModels tersedia: {len(results)}")


In [ ]:
# Tabel A: Konteks literatur (subset berbeda, TIDAK sebanding langsung)
print("TABEL A — Konteks Literatur (subset/protokol berbeda — not directly comparable)")
print(f"  {'Method':<32} {'Subset':<23} {'Dice':>6} {'IoU':>6} {'MAE':>7} {'MSE':>7} {'Params':>8}")
print("  " + "-" * 95)
for row in [
    ("Jeong et al. 2025 (diagnostics)", "CUBS1 (n=2176)",        "0.822", "0.698",    "—",      "—",    "—"),
    ("Sarmun et al. 2024",              "CUBS2 (n=500)",         "0.823", "0.696", "0.166",  "0.049",    "—"),
    ("M³-UNet (Wang et al. 2025)",      "CUBS1 (n=2176, 5-fold)","0.888", "0.799", "0.097",    "—",  "9.31M"),
]:
    print(f"  {row[0]:<32} {row[1]:<23} {row[2]:>6} {row[3]:>6} {row[4]:>7} {row[5]:>7} {row[6]:>8}")
print("\n  * Hanya sebagai konteks; klaim SOTA hanya dari Tabel B di bawah.")


In [ ]:
# Tabel B: Controlled comparison — semua protokol identik
METRICS = ["dice","iou","recall","precision","accuracy","mae_cimt_mm","mse_cimt_mm","params_m","gflops"]

rows_b = []
for model_name, df in results.items():
    row = {"Model": model_name}
    for m in METRICS:
        if m in df.columns:
            vals = df[m].dropna().values
            row[m+"_mean"] = float(vals.mean())
            row[m+"_std"]  = float(vals.std())
        else:
            row[m+"_mean"] = float("nan"); row[m+"_std"] = float("nan")
    row["n_folds"] = len(df)
    rows_b.append(row)

summary_df = pd.DataFrame(rows_b)
summary_df.to_csv("./results/bcvmamba_baseline_summary.csv", index=False)

print("TABEL B — Controlled Comparison (CUBS combined, 5-fold identik, protokol sama)")
hdr2 = f"  {'Model':<30}" + "".join([f"  {m:>14}" for m in ["Dice±std","IoU±std","Recall±std","Prec±std","MAE±std(mm)","MSE±std(mm)"]])
print(hdr2)
print("  " + "-" * 120)
for _, row in summary_df.iterrows():
    parts = [f"  {row['Model']:<30}"]
    for m in ["dice","iou","recall","precision","mae_cimt_mm","mse_cimt_mm"]:
        mn = row[m+"_mean"]; std = row[m+"_std"]
        parts.append(f"  {f'{mn:.3f}±{std:.3f}' if not _math.isnan(mn) else '—':>14}")
    print("".join(parts))

print(f"\n  {'Model':<30}  {'Params(M)':>10}  {'GFLOPs':>8}  {'n_folds':>8}")
for _, row in summary_df.iterrows():
    print(f"  {row['Model']:<30}  {row['params_m_mean']:>10.3f}  {row['gflops_mean']:>8.3f}  {int(row['n_folds']):>8}")


In [ ]:
# Uji statistik: paired t-test + Wilcoxon vs BC-VMamba +L_cimt (usulan)
our_key = next((k for k in results if "ours" in k.lower() or "bc-vmamba" in k.lower()), None)
if our_key and len(results) > 1:
    our_dice = results[our_key]["dice"].values
    print("UJI SIGNIFIKANSI — paired t-test + Wilcoxon signed-rank vs BC-VMamba +L_cimt (usulan)")
    print(f"  {'Model':<30} {'Mean Dice':>10} {'p-t':>10} {'p-W':>10} {'Sig.':>6}")
    print("  " + "-" * 70)
    for name, df in results.items():
        if name == our_key or "dice" not in df.columns: continue
        bl = df["dice"].values
        n  = min(len(our_dice), len(bl))
        if n < 2: continue
        try:
            _, p_t = ttest_rel(our_dice[:n], bl[:n])
            _, p_w = wilcoxon(our_dice[:n] - bl[:n])
        except Exception: p_t = p_w = float("nan")
        sig = "***" if p_t < 0.001 else "**" if p_t < 0.01 else "*" if p_t < 0.05 else "ns"
        pt_s = f"{p_t:.4f}" if not _math.isnan(p_t) else "—"
        pw_s = f"{p_w:.4f}" if not _math.isnan(p_w) else "—"
        print(f"  {name:<30} {bl[:n].mean():>10.4f} {pt_s:>10} {pw_s:>10} {sig:>6}")
    print(f"\n  * p<0.05  ** p<0.01  *** p<0.001  ns=not significant  (n={n} folds)")
else:
    print("Belum cukup hasil untuk uji statistik (perlu BC-VMamba +L_cimt + min 1 baseline).")


In [ ]:
# Visualisasi: Box plot Dice + Bar MAE + Scatter Params vs Dice
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if len(results) < 2:
    print("Belum cukup data untuk visualisasi. Jalankan minimal 2 baseline + BC-VMamba usulan.")
else:
    # Urutkan: ours pertama, sisanya alphabetical
    our_key = next((k for k in results if "ours" in k.lower() or "bc-vmamba" in k.lower()), None)
    ordered = ([our_key] if our_key else []) + [k for k in results if k != our_key]

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    palette = ["steelblue"] + ["#e08c70"] * (len(ordered) - 1)
    short_labels = [n.replace("BC-VMamba +L_cimt (ours)", "BC-VMamba\n+L_cimt★")
                      .replace("VM-UNet baseline", "VM-UNet\nbaseline") for n in ordered]

    # Box plot Dice
    ax = axes[0]
    bp = ax.boxplot([results[n]["dice"].values for n in ordered],
                    labels=short_labels, patch_artist=True)
    for patch, c in zip(bp["boxes"], palette):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    ax.set_title("Distribusi Dice — 5-Fold"); ax.set_ylabel("Dice")
    ax.tick_params(axis='x', labelsize=7); ax.grid(axis='y', alpha=0.3)

    # Bar MAE
    ax = axes[1]
    mae_m = [results[n]["mae_cimt_mm"].mean() if "mae_cimt_mm" in results[n].columns else 0 for n in ordered]
    mae_s = [results[n]["mae_cimt_mm"].std()  if "mae_cimt_mm" in results[n].columns else 0 for n in ordered]
    ax.bar(range(len(ordered)), mae_m, yerr=mae_s, capsize=4, color=palette, alpha=0.75)
    ax.set_xticks(range(len(ordered))); ax.set_xticklabels(short_labels, fontsize=7)
    ax.set_title("MAE CIMT (mm)"); ax.set_ylabel("MAE (mm)"); ax.grid(axis='y', alpha=0.3)

    # Scatter Params vs Dice
    ax = axes[2]
    for name, color in zip(ordered, palette):
        df = results[name]
        if "dice" not in df.columns or "params_m" not in df.columns: continue
        dm = df["dice"].mean(); ds = df["dice"].std(); pm = df["params_m"].mean()
        ax.errorbar(pm, dm, yerr=ds, fmt='o', color=color, capsize=4,
                    markersize=10 if name == our_key else 7)
        lbl = "BC-VMamba +L_cimt★" if name == our_key else name
        ax.annotate(lbl, (pm, dm), xytext=(5, 3), textcoords="offset points", fontsize=7)
    ax.set_xlabel("Params (M)"); ax.set_ylabel("Val Dice (mean)")
    ax.set_title("Efficiency vs Performance"); ax.grid(alpha=0.3)

    plt.suptitle("BC-VMamba — Controlled Baseline Comparison (CUBS combined, 5-fold)", fontsize=11)
    plt.tight_layout()
    plt.savefig("./results/bcvmamba_baseline_comparison.png", dpi=150); plt.show()
    print("Saved: results/bcvmamba_baseline_comparison.png")


### Cara Membaca Hasil

**Tabel B** adalah tabel utama untuk paper (semua protokol identik).
**Tabel A** hanya konteks literatur — tampilkan terpisah dengan catatan "reported in original papers; different subsets/splits".

- `p < 0.05` → perbedaan signifikan; `ns` → dalam noise
- Scatter Params vs Dice: posisi **kiri-atas** = sedikit params + Dice tinggi (ideal)
- BC-VMamba (+L_cimt, 9.63M params) diharapkan kompetitif pada trade-off efisiensi-akurasi vs CNN 30M+ & transformer
